# A3 · The Memory · Harman Singh Sandhu

Built on NB3 (Lecture 3). NB3's cells are kept as they are; my A3 cells sit under headers that start with **A3 ·**, next to the NB3 section they extend. Every output in this notebook is from my own run.


# NB3 · The Memory: retrieving from Northbridge's documents

**Agentic Systems Engineering · Lecture 3 · Vizuara · Dr. Rajat Dandekar**

Northbridge Cycles builds e-bikes in Manchester on **Ledgerline ERP 7.2** (1998). Before we change anything, Henrik
Weber, the plant head, has shared 28 years of documentation as a GitHub repository, `northbridge-docs`. Ask
Northbridge must answer staff questions from it, with the document each answer comes from. Two running examples:

> **Emma:** Can I buy the battery packs from Brightcell Energy? They quote £104.
> **Emma:** What does ERR-PO-0412 mean and how do I fix it?

This notebook follows the lecture:

1. clone the dump and audit it: files, classes, tokens
2. parse it two ways and see what the plain parser loses
3. the questions people ask, and the golden set we score against
4. why pasting everything fails
5. chunking three ways
6. BM25 from scratch, scored on the 224 documentation questions
7. a reranker, then embeddings in Chroma, and why exact codes need BM25
8. the shared answer step: Emma's question answered from RAG's passages (recorded, replayed)
9. the LLM Wiki: a real page, search over its sections, and one query's chain (recorded, replayed)
10. the race, read from the log

**No API key needed.** Retrieval runs here, on your machine. Every model call (answers, the judge, the wiki's
navigation) is replayed from our recorded runs, so you see exactly what happened in the race. Four exercises are
marked `TODO (exercise N)`, each with its solution behind `SOLUTION = True`. Set it to `False` and try it yourself.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: auto


In [2]:
# Lecture 3's extra packages: parsers, the tokenizer, Chroma and the embedding and reranking models (about 2 minutes).
import importlib.util
if importlib.util.find_spec("chromadb") is None or importlib.util.find_spec("pdfplumber") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf==5.1.0", "pdfplumber==0.11.4", "openpyxl",
                    "python-docx", "tiktoken", "scipy", "chromadb==1.0.15", "sentence-transformers", "pytesseract"], check=True)
if "google.colab" in sys.modules and subprocess.run(["which", "tesseract"], capture_output=True).returncode:
    subprocess.run("apt-get -qq install -y tesseract-ocr > /dev/null", shell=True)   # OCR for the six scans
os.environ.setdefault("LEDGER_WIKI_PROVIDER", "google-direct")   # where a live wiki call would go (only if you add a key)
SOLUTION = True

## A3 · Running locally

Parts 2 to 4 run on my own machine (Windows, RTX 3060, 12 GB), so every latency is measured on one machine.
Part 1's outputs above are from Colab. To rerun locally: run the setup cells, the cell below, the imports, the clone
and the parse in section 1, then go straight to **A3 · Part 2** (the A3 Part 1 cells keep their Colab outputs).

In [1]:
# A3 · Running locally (Windows, RTX 3060). Skip on Colab. Run it before the clone and the parse in section 1.
import os
import shutil
import sys

# 1) Tesseract: its installer does not put itself on PATH. Without it the parser reads the six scans as empty and
#    caches that, so this must happen before the first parse.
TESSERACT_DIR = r"C:\Program Files\Tesseract-OCR"
if os.name == "nt" and os.path.isdir(TESSERACT_DIR) and TESSERACT_DIR not in os.environ["PATH"]:
    os.environ["PATH"] = TESSERACT_DIR + os.pathsep + os.environ["PATH"]
print("tesseract:", shutil.which("tesseract") or "NOT FOUND: the six scans will parse as empty")

# 2) Git for Windows turns LF into CRLF on checkout by default. The dump has no .gitattributes, so its e-mails, YAML,
#    CSV and Markdown would change on disk (and with them the token counts). Keep the files byte for byte.
os.environ.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "core.autocrlf", "GIT_CONFIG_VALUE_0": "false"})
print("git:", shutil.which("git"), "| core.autocrlf forced to false for the clone")

# 3) The GPU
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

# 4) Paths. The course code lists files with os.path.relpath, which on Windows gives back-slashes
#    ("01_policies\\HR\\CONFIDENTIAL\\..."), while the golden set and the parser's own checks ("/CONFIDENTIAL/", "/old/")
#    use forward slashes. Unpatched, no retrieved chunk ever matches a gold source (recall 0 everywhere) and the HR
#    folder is not flagged confidential. Return forward-slash paths, as on Linux and Colab.
from ledgerline.course.l3 import corpus as C

if os.name == "nt" and not getattr(C.list_files, "posix_paths", False):
    _list_files = C.list_files

    def list_files(root):
        return sorted(f.replace("\\", "/") for f in _list_files(root))

    list_files.posix_paths = True
    C.list_files = list_files
print("file paths:", "forward slashes (patched for Windows)" if getattr(C.list_files, "posix_paths", False) else "as listed")

tesseract: C:\Program Files\Tesseract-OCR\tesseract.EXE
git: C:\Program Files\Git\cmd\git.EXE | core.autocrlf forced to false for the clone
python 3.11.9 | torch 2.6.0+cu124 | GPU: NVIDIA GeForce RTX 3060
file paths: forward slashes (patched for Windows)


In [2]:
import collections
import json
import pathlib
import statistics
import time

from ledgerline.course.l3 import corpus as C
from ledgerline.course.l3 import evaluate as E
from ledgerline.course.l3 import rag


def pp(o, n=1400):
    """Print JSON, cut at n characters."""
    s = json.dumps(o, indent=1, ensure_ascii=False, default=str)
    print(s if len(s) <= n else s[:n] + " …")

## 1 · Clone the dump, audit it

Henrik's repository is public: `github.com/RajatDandekar/northbridge-docs`. `dump_root` clones it next to this
notebook (or uses `$NORTHBRIDGE_DOCS`, if you set it to a copy you already have).

In [3]:
DOCS = C.dump_root(clone_to="northbridge-docs")
print(DOCS)
print((DOCS / "README.md").read_text()[:900])

C:\Users\harma\OneDrive\Desktop\python\vizuara\Enterprise_Agents\Assignment_A3\northbridge-docs
# northbridge-docs

**From Henrik Weber, plant head, Northbridge Cycles, Manchester**

Hello, and thank you for taking this on.

You asked for our documentation before you change anything, and you were right to. Here it is: everything we could find about how Northbridge buys, stores, builds, sells and pays, in the shape it is really in. We have run Ledgerline since 1998. The rules live in policies, in procedures, in the vendor's manuals, in spreadsheets people keep on the share, and, if I am honest, in Luis's head and in e-mails that never made it into a policy.

Some of it is out of date. Some of it is there twice. A few documents disagree with each other, and a couple are not in force yet. I have not tidied it, because I think you need to see what we actually work from. Where something is old, the newer document usually says so; where it does not, ask Luis or Daniel.

One folder is pri


In [6]:
files = C.list_files(DOCS)
by_folder = collections.Counter(f.split("/")[0] if "/" in f else "(root)" for f in files)
by_class = collections.Counter(C.FMT_CLASS.get(C.fmt_of(f), "other") for f in files)
print(len(files), "files")
for k, v in sorted(by_folder.items()):
    print(f"  {k:24} {v:4}")
print(dict(by_class))

693 files
  (root)                      1
  01_policies                24
  02_procedures              26
  03_ledgerline_manuals      13
  04_service_definitions    405
  05_master_data             15
  06_transactions            26
  07_data_dictionary          4
  08_release_notes           18
  09_email_archive           79
  10_meeting_notes           32
  11_training                34
  12_integrations            10
  99_scans                    6
{'unstructured': 145, 'structured': 27, 'semi-structured': 521}


Files are not the cost. Tokens are. We parse every file with the **structured** parser (one unit per PDF section,
spreadsheet row, service, e-mail) and count tokens with tiktoken `o200k_base`. The first run parses all 693 files
(a few minutes on Colab); the result is cached.

In [7]:
t0 = time.time()
units = C.parse(DOCS, parser="structured")
print(f"{len(units):,} units in {time.time() - t0:.0f} s; failures: {len(C.parse.failures)}")
tok_by_class = collections.Counter()
for u in units:
    tok_by_class[C.FMT_CLASS.get(u.fmt, "other")] += C.n_tokens(u.text)
total_tokens = sum(tok_by_class.values())
for k, v in tok_by_class.most_common():
    print(f"  {k:16} {v:>10,} tokens  {100 * v / total_tokens:4.1f}%")
print(f"  {'total':16} {total_tokens:>10,}")

51,325 units in 1 s; failures: 0
  structured        3,216,916 tokens  69.3%
  unstructured      1,248,302 tokens  26.9%
  semi-structured     179,389 tokens   3.9%
  total             4,644,607


## A3 · Part 1 · The audit

Files, PDF pages, spreadsheet rows and tokens by folder and by class (tiktoken `o200k_base` on the structured parser's text), then the evidence for the mess. These cells reuse `DOCS`, `files`, `units` and `C` from section 1. No model calls.

In [ ]:
# A3 · Part 1 · 1a. Files, PDF pages, spreadsheet rows and tokens, by folder and by class.
# Tokens: tiktoken o200k_base on the structured parser's text, one count per file (the file's units joined by newlines).
# Plain-parser tokens are counted the same way, for comparison.
import collections
import hashlib
from pypdf import PdfReader
from IPython.display import Markdown, display

SHOW_RAW_MARKDOWN = False

plain_units = C.parse(DOCS, parser="plain")   # about 60 s the first time, cached afterwards


def group_by_file(us):
    out = collections.defaultdict(list)
    for u in us:
        out[u.file].append(u)
    return out


S, P = group_by_file(units), group_by_file(plain_units)

per_file = {}
for rel in files:
    fmt = C.fmt_of(rel)
    us = S.get(rel, [])
    per_file[rel] = {
        "folder": rel.split("/")[0] if "/" in rel else "(root)",
        "class": C.FMT_CLASS.get(fmt, "unstructured"),
        "pages": len(PdfReader(DOCS / rel).pages) if fmt == "pdf" else 0,
        "rows": sum(u.kind == "table" for u in us) if fmt in ("xlsx", "csv") else 0,
        "tokens": C.n_tokens("\n".join(u.text for u in us)),
        "plain_tokens": C.n_tokens("\n".join(u.text for u in P.get(rel, []))),
    }

TOTAL_TOKENS = sum(d["tokens"] for d in per_file.values())


def roll_up(key):
    out = collections.defaultdict(collections.Counter)
    for d in per_file.values():
        o = out[d[key]]
        o["files"] += 1
        for k in ("pages", "rows", "tokens", "plain_tokens"):
            o[k] += d[k]
    return dict(sorted(out.items()))


def show_table(md):
    """Render a markdown table as a real table in the notebook; print its source too when SHOW_RAW_MARKDOWN is on."""
    display(Markdown(md))
    if SHOW_RAW_MARKDOWN:
        print(md)


def print_table(key, title):
    rows = roll_up(key)
    lines = [f"| {title} | Files | PDF pages | Rows | Tokens (o200k) | Share of tokens | Plain-parser tokens |",
             "|---|---:|---:|---:|---:|---:|---:|"]
    total = collections.Counter()
    for name, o in rows.items():
        total.update(o)
        lines.append(f"| {name} | {o['files']:,} | {o['pages']:,} | {o['rows']:,} | {o['tokens']:,} | "
                     f"{100 * o['tokens'] / TOTAL_TOKENS:.1f}% | {o['plain_tokens']:,} |")
    lines.append(f"| **Total** | **{total['files']:,}** | **{total['pages']:,}** | **{total['rows']:,}** | "
                 f"**{total['tokens']:,}** | 100% | **{total['plain_tokens']:,}** |")
    show_table("\n".join(lines))

print_table("folder", "Folder")
print_table("class", "Class")
print(f"Structured parse / plain parse: {TOTAL_TOKENS / sum(d['plain_tokens'] for d in per_file.values()):.3f}x")

| Folder | Files | PDF pages | Rows | Tokens (o200k) | Share of tokens | Plain-parser tokens |
|---|---:|---:|---:|---:|---:|---:|
| (root) | 1 | 0 | 0 | 780 | 0.0% | 783 |
| 01_policies | 24 | 58 | 8 | 32,181 | 0.7% | 33,193 |
| 02_procedures | 26 | 116 | 0 | 55,512 | 1.2% | 58,921 |
| 03_ledgerline_manuals | 13 | 496 | 0 | 288,094 | 6.1% | 302,991 |
| 04_service_definitions | 405 | 0 | 400 | 162,467 | 3.5% | 155,669 |
| 05_master_data | 15 | 0 | 18,592 | 1,126,597 | 24.0% | 645,745 |
| 06_transactions | 26 | 483 | 25,353 | 2,892,604 | 61.6% | 2,009,767 |
| 07_data_dictionary | 4 | 27 | 1,188 | 43,306 | 0.9% | 25,982 |
| 08_release_notes | 18 | 53 | 9 | 29,125 | 0.6% | 30,865 |
| 09_email_archive | 79 | 0 | 0 | 19,419 | 0.4% | 26,772 |
| 10_meeting_notes | 32 | 0 | 0 | 19,268 | 0.4% | 18,675 |
| 11_training | 34 | 26 | 0 | 16,342 | 0.3% | 27,206 |
| 12_integrations | 10 | 8 | 7 | 5,999 | 0.1% | 6,615 |
| 99_scans | 6 | 6 | 0 | 534 | 0.0% | 0 |
| **Total** | **693** | **1,273** | **45,557** | **4,692,228** | 100% | **3,343,184** |

| Class | Files | PDF pages | Rows | Tokens (o200k) | Share of tokens | Plain-parser tokens |
|---|---:|---:|---:|---:|---:|---:|
| semi-structured | 521 | 0 | 0 | 179,398 | 3.8% | 200,339 |
| structured | 27 | 0 | 45,557 | 3,262,429 | 69.5% | 1,914,476 |
| unstructured | 145 | 1,273 | 0 | 1,250,401 | 26.6% | 1,228,369 |
| **Total** | **693** | **1,273** | **45,557** | **4,692,228** | 100% | **3,343,184** |

Structured parse / plain parse: 1.404x


In [ ]:
# A3 · Part 1 · 1b. Evidence for the mess, found from the documents themselves (no file names typed in).
import re

# 1) Byte-identical files: the same SHA-1 hash
sha = collections.defaultdict(list)
for rel in files:
    sha[hashlib.sha1((DOCS / rel).read_bytes()).hexdigest()].append(rel)
print("Byte-identical files:")
for group in sha.values():
    if len(group) > 1:
        print("  ", group)

# 2) The parser's own flags, read from each document's status, effective date and folder
meta = {u.file: u.meta for u in units}
for flag in ("superseded", "future", "confidential"):
    flagged = sorted(f for f, m in meta.items() if m.get(flag))
    print(f"\n{flag}: {len(flagged)} files")
    for f in flagged:
        print("  ", f, "| status:", meta[f].get("status") or "-", "| effective:", meta[f].get("effective") or "-")


# 3) One document, several files: group by the document code the parser read from each file
def state_of(m):
    return "superseded" if m.get("superseded") else "not yet in force" if m.get("future") else "current"


by_doc = collections.defaultdict(set)
for u in units:
    if u.meta.get("doc_id"):
        by_doc[u.meta["doc_id"]].add(u.file)
versions = {d: sorted(fs, key=lambda f: meta[f].get("version") or "") for d, fs in sorted(by_doc.items()) if len(fs) > 1}
print(f"\nDocuments that exist as more than one file: {len(versions)}")
for d, fs in versions.items():
    print(f"  {d}")
    for f in fs:
        m = meta[f]
        print(f"     v{m.get('version') or '?':7} {state_of(m):17} effective {m.get('effective') or '-':10}  {f}")

# 4) What changed between versions: today's version's numbered clauses ("7.2 Any order ...") against each other
#    version's, matched by clause number. PDFs wrap lines and DOCX does not, so two clauses are compared only as far
#    as both texts run; a space the PDF reader puts before a comma or full stop is ignored.
CLAUSE = re.compile(r"^\s*(\d+(?:\.\d+)+)\s+([A-Z].*)$")


def clauses(rel):
    out = {}
    for u in S[rel]:
        for line in u.text.splitlines():
            m = CLAUSE.match(line)
            if m and m.group(1) not in out:
                out[m.group(1)] = re.sub(r"\s+([,.;:)])", r"\1", " ".join(m.group(2).split()))
    return out


def differs(a, b, n=120):
    k = min(len(a), len(b), n)
    return a[:k] != b[:k]


print("\nClauses that differ between today's version of a document and its other versions")
for d, fs in versions.items():
    current = [f for f in fs if state_of(meta[f]) == "current"]
    if not current:
        continue
    cur = current[-1]
    cc = clauses(cur)
    for other in fs:
        if other == cur:
            continue
        oc = clauses(other)
        changed = [n for n in cc if n in oc and differs(cc[n], oc[n])]
        added = [n for n in cc if n not in oc]
        if not (changed or added):
            continue   # the same clauses: a copy, or a file without numbered clauses
        m = meta[other]
        label = f"from {m.get('effective')}" if m.get("future") else "before"
        print(f"\n--- {d}: today v{meta[cur].get('version')} ({cur.split('/')[-1]})"
              f"  |  {label}: v{m.get('version')}, {state_of(m)} ({other.split('/')[-1]})")
        if added:
            print("   only in today's version: §" + ", §".join(added[:15]))
        for n in changed[:8]:
            print(f"   §{n} {label + ':':17} {oc[n][:140]}")
            print(f"   §{n} {'today:':17} {cc[n][:140]}")

# 5) Scans: tokens under the plain parser against the structured parser's OCR
print("\nScans: plain-parser tokens -> structured (OCR) tokens")
for rel in sorted(f for f in files if f.startswith("99_scans/")):
    print(f"  {per_file[rel]['plain_tokens']:5} -> {per_file[rel]['tokens']:5}  {rel}")

Byte-identical files:
   ['01_policies/POL-SUP-003 Supplier Policy v2.3 (1).pdf', '01_policies/POL-SUP-003 Supplier Policy v2.3.pdf']

superseded: 14 files
   01_policies/old/Approval_Matrix_v2.0_2024.docx | status: SUPERSEDED | effective: 2024-04-01
   01_policies/old/Purchase Policy v4.0.docx | status: SUPERSEDED | effective: 2025-04-01
   01_policies/old/Supplier_Policy_v2.2.docx | status: SUPERSEDED | effective: 2025-04-01
   01_policies/old/purchase_policy_v3_2019.docx | status: SUPERSEDED | effective: 2019-04-01
   02_procedures/SOP-PUR-002 PR to PO v1.3 (FINAL).pdf | status: SUPERSEDED | effective: 2023-05-01
   02_procedures/old/SOP-PUR-001 Supplier onboarding v2.0 (2021).docx | status: SUPERSEDED | effective: 2021-02-01
   03_ledgerline_manuals/old/LL6_purchasing_quickguide.docx | status: SUPERSEDED | effective: 2009-05-01
   03_ledgerline_manuals/old/Ledgerline 7.0 Purchasing Manual (2015).pdf | status: SUPERSEDED | effective: 2015-03-02
   03_ledgerline_manuals/old/MAN-INV-7

## 2 · Two parsers, one file

The **plain** parser takes a file's text as it comes. The **structured** parser reads each format properly: e-mail
headers decoded, every spreadsheet row written as `column: value`, scans read by OCR.

In [ ]:
for rel in ["09_email_archive/2026/2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml",
            "99_scans/2019_Voltway_code_change_approval_signed.pdf"]:
    for p in ("plain", "structured"):
        us = C.parse_file(DOCS, rel, parser=p)
        text = us[0].text if us else ""
        print(f"--- {p:10} {rel.split('/')[-1]}: {len(us)} unit(s), {C.n_tokens(text)} tokens")
        print(text[:420] or "(no text: an image-only page)")

--- plain      2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml: 1 unit(s), 310 tokens
From: Julien Marchand <j.marchand@brightcell-energy.example>
To: Emma Clarke <emma.clarke@northbridgecycles.example>
Date: Thu, 22 Oct 2026 16:41:00 +0000
Subject: Quotation Q-BE-26-0917 - 36 V 500 Wh battery packs
Message-ID: <c0e8939f459f7898bbeb@mail.northbridgecycles.example>
X-Mailer: NBC Mail Gateway 4.2
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: quoted-printable
MIME-Version: 
--- structured 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml: 1 unit(s), 229 tokens
E-mail. Subject: Quotation Q-BE-26-0917 - 36 V 500 Wh battery packs
From: Julien Marchand <j.marchand@brightcell-energy.example>
To: Emma Clarke <emma.clarke@northbridgecycles.example>
Date: Thu, 22 Oct 2026 16:41:00 +0000

Dear Ms Clarke,

Thank you for your call. As promised, our quotation Q-BE-26-0917 for battery packs, 36 V 500 Wh, compatible with your BAT-500 mounting: £104.00 per pack, 

In [ ]:
rows = {p: C.parse_file(DOCS, "05_master_data/SUPPLIER_MASTER.xlsx", parser=p) for p in ("plain", "structured")}
for p, us in rows.items():
    u = next(x for x in us if "VC-EIN-0042" in x.text)
    print(f"{p:10} {u.loc}  {u.text[:240]}")

plain      {'sheet': 'Suppliers', 'row': 4}  VC-EIN-0042 Voltway Cells NL859312774B01 NL Eindhoven ACTIVE N45 BAT MIGRATION 2019-04-01 Femke de Vries orders@vc-ein-0042.example
structured {'sheet': 'Suppliers', 'row': 4}  sup_cd: VC-EIN-0042; name: Voltway Cells; vat_no: NL859312774B01; state: NL; city: Eindhoven; status: ACTIVE; pt_cd: N45; supplies: BAT; created_by: MIGRATION; created_on: 2019-04-01; contact: Femke de Vries; email: orders@vc-ein-0042.examp


In [ ]:
# A3 · Part 1 · 2. Two parsers, one file: the rate contract for BAT-500, and the question it breaks (Q-402).
# Q-402 "What is Voltway Cells' contract rate for BAT-500 now?" has two gold rows in sheet 2026-27:
# row 1449 (£114.00, until 30 Sep 2026) and row 1450 (£119.00, from 1 Oct 2026). Its answer must contain "119.00".
import numpy as np

RC = "05_master_data/Rate Contracts 2025-26 and 2026-27.xlsx"
GOLD_ROWS = (1449, 1450)
q402 = next(q for q in C.questions(DOCS) if q["id"] == "Q-402")
print("Q-402:", q402["question"], "| must contain:", q402["must_contain"], "\n")

# The two gold rows, as each parser writes them
for p in ("plain", "structured"):
    for u in C.parse_file(DOCS, RC, parser=p):
        if u.loc.get("sheet") == "2026-27" and u.loc.get("row") in GOLD_ROWS:
            print(f"--- {p:10} row {u.loc['row']} | '119.00' visible: {'119.00' in u.text}")
            print("   ", u.text[:300])
    print()

# Does search still find the row? The same chunker (structure-aware, 200 tokens, 15% overlap) and the same BM25
# over the whole dump, once from the plain parse and once from the structured parse. Emma's role, so HR is excluded.
def gold_rank(chunks, hits):
    for r, (i, _) in enumerate(hits, 1):
        c = chunks[i]
        a, b = (c.locs.get("rows") or {}).get("2026-27", (0, 0))
        if c.file == RC and any(a <= g <= b for g in GOLD_ROWS):
            return r
    return None

for p, us in (("plain", plain_units), ("structured", units)):
    ch = C.chunk(us, "structure", 200, 0.15)
    bm = rag.BM25([c.text for c in ch])
    allowed = np.array([not c.meta.get("confidential") for c in ch])
    hits = bm.search(q402["question"], 50, allowed)
    print(f"=== {p}: {len(ch):,} chunks | first gold row at rank {gold_rank(ch, hits) or 'not in top 50'}")
    for r, (i, s) in enumerate(hits[:5], 1):
        print(f"  {r}. {s:6.2f}  {ch[i].cite}")
    print()

Q-402: What is Voltway Cells' contract rate for BAT-500 now? | must contain: ['119.00'] 

--- plain      row 1449 | '119.00' visible: False
    VC-EIN-0042/BAT-500/2026-04-01 VC-EIN-0042 BAT-500 114 2026-04-01 2026-09-30
--- plain      row 1450 | '119.00' visible: False
    VC-EIN-0042/BAT-500/2026-10-01 VC-EIN-0042 BAT-500 119 2026-10-01 2027-03-31

--- structured row 1449 | '119.00' visible: False
    line: VC-EIN-0042/BAT-500/2026-04-01; sup_cd: VC-EIN-0042; itm_cd: BAT-500; rate: 114.00; valid_from: 2026-04-01; valid_to: 2026-09-30
--- structured row 1450 | '119.00' visible: True
    line: VC-EIN-0042/BAT-500/2026-10-01; sup_cd: VC-EIN-0042; itm_cd: BAT-500; rate: 119.00; valid_from: 2026-10-01; valid_to: 2027-03-31

=== plain: 19,634 chunks | first gold row at rank not in top 50
  1.  39.91  2026-09-25_FW_ Voltway Cells - BAT-500 price from 1 October 2026.eml
  2.  33.87  2026-09-25_FW_ Voltway Cells - BAT-500 price from 1 October 2026.eml
  3.  32.66  2026-10-19 Weekly purchasing

## 3 · The questions people ask

The golden set has 304 real-style questions in three kinds. **Genesis** (create something) and **analytical**
(count, sum, compare) need services and tools: Lecture 4 onward. **Documentation** questions are today's.

In [ ]:
allq = C.questions(DOCS, category="all")
print(collections.Counter(q["category"] for q in allq))
G = E.Golden(DOCS)
print(len(G.qs), "documentation questions:", dict(collections.Counter(q["type"] for q in G.qs)))
q412 = G.by_id["Q-0412"]
pp({k: q412[k] for k in ("question", "reference_answer", "gold_sources", "must_contain")})

Counter({'documentation': 224, 'genesis': 40, 'analytical': 40})
224 documentation questions: {'multi_hop': 21, 'exact_code': 27, 'single_lookup': 23, 'procedure': 20, 'rule': 25, 'service_field': 32, 'table_lookup': 34, 'version_conflict': 20, 'not_in_docs': 12, 'permission': 10}
{
 "question": "What does ERR-PO-0412 mean and how do I fix it?",
 "reference_answer": "ERR-PO-0412 TAX CODE DOES NOT MATCH SUPPLIER COUNTRY: the tax code on the order is for the wrong country (the supplier's country code is shown in brackets). A UK supplier needs a -UK code; a supplier in any other country needs a reverse-charge -RC code. Fetch always fills VAT20-UK, so for an overseas supplier such as Voltway Cells (NL) change the tax code to the -RC code with the same rate, for example VAT20-RC, and save again.",
 "gold_sources": [
  {
   "file": "03_ledgerline_manuals/MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf",
   "fact_id": "F-MAN-17.2",
   "page": 10,
   "section": "17.2 ERR-PO-0412 

## 4 · Why not paste everything?

Gemini 3.8 Flash's window is 1,048,576 tokens (OpenRouter's listing on 3 Nov 2026; the largest window listed was
2,000,000). Our dump is:

In [ ]:
print(f"{total_tokens:,} tokens = {total_tokens / 1_048_576:.1f} x Gemini 3.8 Flash's window, {total_tokens / 2_000_000:.1f} x the largest")
print(f"input cost of one question, if it fitted: ${total_tokens * 0.825 / 1e6:.2f} (at $0.825 per million input tokens)")

4,644,608 tokens = 4.4 x Gemini 3.8 Flash's window, 2.3 x the largest
input cost of one question, if it fitted: $3.83 (at $0.825 per million input tokens)


## 5 · Chunking, three ways

A chunk is what gets indexed and searched. Fixed windows cut wherever 200 tokens end; recursive splitting prefers
paragraph and line breaks; structure-aware chunking packs whole sections and never cuts one unless it must.

In [ ]:
policy = [u for u in units if u.file == "01_policies/POL-PUR-001_Purchase_Policy_v4.1.pdf"]
for m in ("fixed", "recursive", "structure"):
    ch = C.chunk(policy, m, 200, 0.0)
    print(f"--- {m}: {len(ch)} chunks")
    for c in ch[2:4]:
        print(f"  [{c.cite}] {c.n_tokens} tokens: {c.body[:110]!r}")

--- fixed: 11 chunks
  [POL-PUR-001 v4.1, pp.1-2, § 3. Requests before orders] 235 tokens: '3 A direct purchase order without a request is allowed only when its gross value, including VAT, is\n£50,000 or'
  [POL-PUR-001 v4.1, p.2, § 5. Approval limits] 234 tokens: 'approved by the next person up the line whose limit covers it.\n5.3 Nobody may approve an order they raised.\n5.'
--- recursive: 12 chunks
  [POL-PUR-001 v4.1, p.1, § 3. Requests before orders] 213 tokens: '3.1 Every purchase starts with a purchase request (PR) in Ledgerline, except a direct order under §3.3.\n3.2 A '
  [POL-PUR-001 v4.1, pp.1-2, § 4. Cost centres and budgets] 217 tokens: "4.3 Before raising an order, check the cost centre's status in Ledgerline (screen FIN-CC-01) or ask\nFinance.\n5"
--- structure: 15 chunks
  [POL-PUR-001 v4.1, p.1, § 1. Purpose] 225 tokens: '1. Purpose\nThis policy sets out how Northbridge Cycles buys materials, components and services, so that every\n'
  [POL-PUR-001 v4.1, p.1, § 4. Cost

In [ ]:
t0 = time.time()
chunks = C.chunk(units, "structure", 200, 0.15)   # the race winner's chunking: structured-structure-200-o15
print(f"{len(chunks):,} chunks in {time.time() - t0:.0f} s, {statistics.mean(c.n_tokens for c in chunks):.0f} tokens on average")
by_id = {c.cid: c for c in chunks}

28,953 chunks in 8 s, 196 tokens on average


## 6 · BM25 from scratch

BM25 scores a chunk by the query words it contains, weighted by how rare each word is (idf) and damped for long
chunks. Our tokenizer keeps codes like `ERR-PO-0412` whole. Here is the scoring code we use, then a search.

In [1]:
import inspect
print(inspect.getsource(rag.tokenize))
print(rag.tokenize("What does ERR-PO-0412 mean? Rule 6.2, UN 38.3, BAT-500"))

NameError: name 'rag' is not defined

In [ ]:
idx = rag.Index("/tmp/nb3_index", chunks)
t0 = time.time()
bm = idx.bm25
print(f"BM25 over {len(chunks):,} chunks built in {time.time() - t0:.1f} s, {len(bm.vocab):,} terms")
t0 = time.perf_counter()
hits = bm.search("What does ERR-PO-0412 mean and how do I fix it?", 5, ~idx.conf)
print(f"search: {1000 * (time.perf_counter() - t0):.1f} ms")
for i, s in hits:
    print(f"  {s:6.2f}  {chunks[i].cite}")

Now score BM25 alone on all 224 documentation questions: is a gold source in the top 5? (`not_in_docs` and refused
permission questions are left out of retrieval scoring, so 205 count.)

In [ ]:
bm25_cfg = rag.Config(parser="structured", chunker="structure", size=200, overlap=0.15, retrieval="bm25", rerank=False, k=10)
bm25_only = rag.Retriever(idx, bm25_cfg)
rep = E.retrieval_only(lambda q, role: bm25_only.search(q, role=role), G)
s = rep["summary"]["overall"]
print(f"BM25 alone: recall@5 {s['recall@5']:.3f}, recall@10 {s['recall@10']:.3f}, MRR {s['mrr']:.3f}; "
      f"search p95 {1000 * rep['summary']['latency_s']['p95']:.1f} ms")

The grid measured this exact configuration (structured, structure-aware 200 tokens, 15% overlap, BM25 alone) at
recall@5 0.771 and recall@10 0.873: the same numbers, from a fresh clone.

### TODO (exercise 1) · Which question types does BM25 miss?
Print recall@5 for each question type from `rep["summary"]["by_type"]`, worst first.

In [ ]:
if SOLUTION:
    for t, v in sorted(rep["summary"]["by_type"].items(), key=lambda x: x[1].get("recall@5") or 9):
        if v.get("recall@5") is not None:
            print(f"  {t:18} n={v['n']:3}  recall@5 {v['recall@5']:.2f}")

## 7 · The reranker, then embeddings in Chroma

A **cross-encoder** reranker (`BAAI/bge-reranker-base`) reads the question and each candidate together and scores
the pair. It is slower than BM25, so it only reorders BM25's top 20. This is the race winner's configuration.

In [ ]:
best_cfg = rag.best_config()
print(best_cfg.id)
best = rag.Retriever(idx, best_cfg)
for qid in ("Q-0412", "Q-EMMA"):
    q = G.by_id[qid]
    out = best.search(q["question"], role=C.role_of(q))
    print(f"--- {qid}: {q['question']}  ({1000 * out['seconds']:.0f} ms on this machine; timing {out['timing']})")
    for it in out["items"][:5]:
        print(f"  {it['rank']:2}. {it['cite']}")

On a CPU the cross-encoder takes seconds per question; the race ran it on an L4 GPU, where the whole search took
48 ms at p95. On 40 questions, how much does it add over BM25 alone? (On a CPU this takes a few minutes.)

In [ ]:
sub = G.qs[::5][:40]
a = E.retrieval_only(lambda q, role: bm25_only.search(q, role=role), G, sub)["summary"]["overall"]
b = E.retrieval_only(lambda q, role: best.search(q, role=role), G, sub)["summary"]["overall"]
print(f"40 questions: recall@5 BM25 {a['recall@5']:.3f} -> BM25 + reranker {b['recall@5']:.3f}")

**Embeddings.** A dense retriever turns every chunk into a vector (here `bge-small`, 384 numbers) and finds the
nearest vectors to the question's. We embed a small part of the dump (the manuals, the training cards and the
policies) into a persistent **Chroma** collection, with HNSW set to search widely (`ef_search` 4000, as we measured).

In [ ]:
import chromadb
part = [c for c in chunks if c.file.split("/")[0] in ("01_policies", "03_ledgerline_manuals", "11_training") and not c.meta.get("confidential")]
emb = rag.Embedder("bge-small")
t0 = time.time()
vecs = emb.passages([c.text for c in part])
print(f"{len(part):,} chunks embedded in {time.time() - t0:.0f} s, vectors {vecs.shape}")
client = chromadb.PersistentClient(path="/tmp/nb3_chroma")
col = client.get_or_create_collection("bge-small", configuration={"hnsw": dict(rag.HNSW)})   # cosine, ef_search 4000
if col.count() == 0:
    for i in range(0, len(part), 4000):
        col.add(ids=[str(j) for j in range(i, min(i + 4000, len(part)))], embeddings=vecs[i:i + 4000].tolist())
print(col.count(), "vectors in Chroma")

In [ ]:
def dense(question, k=5):
    r = col.query(query_embeddings=[emb.query(question).tolist()], n_results=k, include=["distances"])
    return [(part[int(i)].cite, round(1 - d, 3)) for i, d in zip(r["ids"][0], r["distances"][0])]

for question in ["What does ERR-PO-0412 mean and how do I fix it?", "Can we buy battery packs from a supplier without a UN 38.3 certificate?"]:
    print("---", question)
    for cite, sim in dense(question):
        print(f"  {sim:.3f}  {cite}")

### TODO (exercise 2) · BM25 against dense on the same part of the dump
Build a BM25 over `part` (`rag.BM25([c.text for c in part])`) and print its top 5 for the ERR-PO-0412 question
next to the dense top 5. Which one finds QRC-03 first?

In [ ]:
if SOLUTION:
    bm_part = rag.BM25([c.text for c in part])
    for i, s in bm_part.search("What does ERR-PO-0412 mean and how do I fix it?", 5):
        print(f"  {s:6.2f}  {part[i].cite}")

## A3 · Part 2 · RAG, tuned

Retrieval only: no answer model is called, so Part 2 costs $0. Everything runs on one machine (RTX 3060).

**The selection rule, fixed before the grid runs:** among configurations whose search p95 is at most **200 ms**,
choose the highest **recall@5**. An exact tie goes to the higher MRR, then the lower p95.

**The grid:** parser (plain, structured) × chunker (fixed, recursive, structure) × size (200, 400, 800 tokens) ×
search (BM25, dense, hybrid) × reranker (off, on) = **108 configurations**. Held fixed: 15% overlap, `bge-small`
embeddings in Chroma with `rag.HNSW` (ef_search 4000), top 10, `bge-reranker-base` over the top 20 when it is on.

In [6]:
# A3 · Part 2 · 1. The golden subset: 40 of the 224 documentation questions, 4 of each of the 10 types.
# Emma's two running questions (Q-EMMA, multi-hop; Q-0412, exact code) are always in; the rest of each type is a
# seeded random draw, so every run gets the same 40. Parts 3 and 4 use the same 40.
import random

from IPython.display import Markdown, display


def show_table(md):
    display(Markdown(md))


G = E.Golden(DOCS)
SEED, PER_TYPE, ALWAYS = 3, 4, ("Q-EMMA", "Q-0412")
by_type = collections.defaultdict(list)
for q in G.qs:
    by_type[q["type"]].append(q)
SUBSET = []
for t in sorted(by_type):
    fixed = [q for q in by_type[t] if q["id"] in ALWAYS]
    rest = [q for q in by_type[t] if q["id"] not in ALWAYS]
    SUBSET += fixed + random.Random(f"{SEED}-{t}").sample(rest, PER_TYPE - len(fixed))
SCORED = [q for q in SUBSET if G.expected(q) == "answer"]

# Check before anything is scored: every gold file must be one of the parsed files, and the HR folder must be flagged
# confidential. (Both failed on Windows before the path patch in "A3 · Running locally".)
parsed_files = {u.file for u in units}
missing = sorted({g["file"] for q in SCORED for g in G.gold(q)} - parsed_files)
assert not missing, f"gold files are not among the parsed files (a path problem?): {missing[:3]}"
n_conf = len({u.file for u in units if u.meta.get("confidential")})
assert n_conf == 4, f"expected the 4 HR files to be flagged confidential, found {n_conf}"
print(f"check passed: all gold files parsed, {n_conf} confidential files flagged")


def gold_label(g):
    where = (f"p.{g['page']}" if g.get("page") else f"pp.{g['pages'][0]}-{g['pages'][-1]}" if g.get("pages")
             else f"{g.get('sheet', '')} row {g['row']}" if g.get("row") else "")
    return f"{g['file'].split('/')[-1]} {where}".strip()


lines = ["| # | Id | Type | Expected | Question | Gold sources |", "|---:|---|---|---|---|---|"]
for n, q in enumerate(SUBSET, 1):
    golds = "; ".join(gold_label(g) for g in G.gold(q)) if G.expected(q) == "answer" else "(scored on the answer)"
    lines.append(f"| {n} | {q['id']} | {q['type']} | {G.expected(q)} | {q['question']} | {golds} |")
show_table("\n".join(lines))
gold_files = sorted({g["file"] for q in SCORED for g in G.gold(q)})
print(f"{len(SUBSET)} questions, {len(SCORED)} scored for recall (one question = {1 / len(SCORED):.3f}); "
      f"{len(gold_files)} distinct gold files:", dict(collections.Counter(f.split("/")[0] for f in gold_files)))

check passed: all gold files parsed, 4 confidential files flagged


| # | Id | Type | Expected | Question | Gold sources |
|---:|---|---|---|---|---|
| 1 | Q-0412 | exact_code | answer | What does ERR-PO-0412 mean and how do I fix it? | MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.10; MAN-ERR-72 Message and Error Code Reference v7.2.3.pdf pp.17-18; MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.7; QRC-03 ERR-PO-0412 quick fix.pdf p.1 |
| 2 | Q-705 | exact_code | answer | What does ERR-SUP-0118 mean? | MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.11 |
| 3 | Q-703 | exact_code | answer | What does ERR-PO-0610 mean? | MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.10 |
| 4 | Q-711 | exact_code | answer | What does ERR-SEC-0001 say? | MAN-ERR-72 Message and Error Code Reference v7.2.3.pdf p.23 |
| 5 | Q-EMMA | multi_hop | answer | Can I buy the battery packs from Brightcell Energy? They quote £104. | POL-PUR-001_Purchase_Policy_v4.1.pdf p.2; SOP-PUR-001 Supplier Onboarding v3.1.pdf p.3; SOP-PUR-001 Supplier Onboarding v3.1.pdf p.3; SOP-PUR-001 Supplier Onboarding v3.1.pdf p.3; AMD-SUP-2026-02 Amendment 2 Supplier Policy.pdf p.1; POL-PUR-001_Purchase_Policy_v4.1.pdf p.2; RN-7.2.3-P4 Ledgerline Patch Notes.pdf p.2; 2026-10-07 CAB minutes (CR-2026-031 certificate check).docx; 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml; 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml; Rate Contracts 2025-26 and 2026-27.xlsx 2026-27 row 1450 |
| 6 | Q-502 | multi_hop | answer | Can I order BAT-500 from Nordic Cell? They're cheaper. | Rate Contracts 2025-26 and 2026-27.xlsx 2026-27 row 848; SUPPLIER_MASTER.xlsx Certificates row 6; POL-PUR-001_Purchase_Policy_v4.1.pdf p.2; 2026-10-01_CERT EXPIRED - DO NOT ORDER_ Nordic Cell Co. (NC-GDA-0063).eml; 2026-09-29_UN 38.3 report renewal.eml |
| 7 | Q-510 | multi_hop | answer | Can I raise a direct order for £40,000 of brake pads next month? | POL-PUR-001_Purchase_Policy_v4.1.pdf p.1; AMD-PUR-2026-03_Amendment3_PurchasePolicy_DIRlimit.pdf p.1; AMD-PUR-2026-03_Amendment3_PurchasePolicy_DIRlimit.pdf p.1; ITEM_MASTER_2026-10-26.xlsx Items row 13 |
| 8 | Q-505 | multi_hop | answer | Does Ledgerline stop me ordering from a supplier without a certificate? | MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.8; RN-7.2.3-P4 Ledgerline Patch Notes.pdf p.2; 2026-10-07 CAB minutes (CR-2026-031 certificate check).docx; POL-SUP-003 Supplier Policy v2.3.pdf p.2; MAN-ADM-72 Ledgerline 7.2 System Administration Guide v7.2.3.pdf p.5 |
| 9 | Q-811 | not_in_docs | abstain | What discount does Voltway give for orders above 5,000 packs? | (scored on the answer) |
| 10 | Q-809 | not_in_docs | abstain | When will Ledgerline 8 be released? | (scored on the answer) |
| 11 | Q-808 | not_in_docs | abstain | Who is the company's data protection officer? | (scored on the answer) |
| 12 | Q-802 | not_in_docs | abstain | What is the Wi-Fi password for the Leeds unit? | (scored on the answer) |
| 13 | Q-907 | permission | answer | As HR manager, what is the senior buyer salary band? | HR-CONF-007 Salary Bands 2026-27.pdf p.1 |
| 14 | Q-904 | permission | refuse | What was the outcome of the disciplinary case about the dropped battery pack? | (scored on the answer) |
| 15 | Q-903 | permission | refuse | Is Leeds stores going to close? | (scored on the answer) |
| 16 | Q-906 | permission | refuse | What percentage bonus can purchasing get? | (scored on the answer) |
| 17 | Q-110 | procedure | answer | How do I raise a purchase request? | SOP-PUR-002 PR to PO v1.4.pdf p.3; SOP-PUR-002 PR to PO v1.4.pdf p.3 |
| 18 | Q-104 | procedure | answer | What do I do if the screen hangs after I press Save on an order? | MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.13 |
| 19 | Q-112 | procedure | answer | How do I raise an order after the request is approved? | SOP-PUR-002 PR to PO v1.4.pdf p.3; SOP-PUR-002 PR to PO v1.4.pdf p.3; SOP-PUR-002 PR to PO v1.4.pdf p.3; SOP-PUR-002 PR to PO v1.4.pdf p.3 |
| 20 | Q-118 | procedure | answer | How are battery packs issued to production? | SOP-PRD-001 Work Order Release and Backflush v1.5.pdf p.3 |
| 21 | Q-219 | rule | answer | What is the warranty on a frame and on a battery pack? | POL-RET-006 Returns Policy v2.1.pdf p.2 |
| 22 | Q-222 | rule | answer | How long are supplier certificates kept? | POL-DRP-009 Data Retention Policy v1.4.pdf p.3 |
| 23 | Q-208 | rule | answer | Does an expired certificate still count? | POL-PUR-001_Purchase_Policy_v4.1.pdf p.2; POL-SUP-003 Supplier Policy v2.3.pdf p.2 |
| 24 | Q-213 | rule | answer | Who approves an order above £5,000,000? | POL-APR-002 Approval Matrix v3.0.pdf p.1 |
| 25 | Q-312 | service_field | answer | Which error does sup_create raise for a duplicate? | sup_create.yaml; sup_create.yaml |
| 26 | Q-322 | service_field | answer | What does the status bar say after po_save? | po_save.yaml; MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.8 |
| 27 | Q-334 | service_field | answer | What are the required fields of tax_compute? | tax_compute.yaml |
| 28 | Q-335 | service_field | answer | What are the required fields of po_hist? | po_hist.yaml |
| 29 | Q-002 | single_lookup | answer | What is the extension for the IT service desk? | POL-ITS-008 IT Security Policy v3.2.pdf p.5; MAN-SCR-72 Screen Reference v7.2.3.pdf p.42 |
| 30 | Q-011 | single_lookup | answer | What is the name of the battery handling course and how often must it be repeated? | POL-BAT-004 Battery Safety and Handling Policy v1.2.pdf p.4 |
| 31 | Q-010 | single_lookup | answer | When does the nightly backup run? | MAN-BAT-72 Batch Jobs and Scheduler Reference (NBC schedule).pdf p.1; MAN-ADM-72 Ledgerline 7.2 System Administration Guide v7.2.3.pdf p.10 |
| 32 | Q-020 | single_lookup | answer | When is the annual stock count? | SOP-INV-003 Stock Count v1.2.pdf p.3 |
| 33 | Q-412 | table_lookup | answer | What is the rate for CHG-4A-01 from Castlefield Chargers? | Rate Contracts 2025-26 and 2026-27.xlsx 2026-27 row 175 |
| 34 | Q-422 | table_lookup | answer | What is the warehouse WH-MAN-BAT? | Warehouses.csv whs row 5 |
| 35 | Q-413 | table_lookup | answer | What is the rate for MTR-HUB-250? | Rate Contracts 2025-26 and 2026-27.xlsx 2026-27 row 21 |
| 36 | Q-433 | table_lookup | answer | What is the reorder level of LGT-REB and is it a lithium-cell part? | ITEM_MASTER_2026-10-26.xlsx Items row 415 |
| 37 | Q-618 | version_conflict | answer | How long do I have to submit an expense claim? | POL-TEX-007 Travel and Expenses Policy v2.4.pdf p.4; POL-TEX-007 Travel and Expenses Policy v3.0 (effective 2027-01-01).pdf p.2 |
| 38 | Q-610 | version_conflict | answer | What hotel rate can I book in London? | POL-TEX-007 Travel and Expenses Policy v2.4.pdf p.2; POL-TEX-007 Travel and Expenses Policy v3.0 (effective 2027-01-01).pdf p.2; POL-TEX-007 Travel and Expenses Policy v3.0 (effective 2027-01-01).pdf p.1 |
| 39 | Q-615 | version_conflict | answer | Did chargers always need UN 38.3? | Purchase Policy v4.0.docx; POL-PUR-001_Purchase_Policy_v4.1.pdf p.2 |
| 40 | Q-607 | version_conflict | answer | Is Fetch optional on the order screen? | RN-7.2.3 Ledgerline Release Notes.pdf p.2; MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf p.7; Ledgerline 7.0 Purchasing Manual (2015).pdf p.2; SOP-PUR-002 PR to PO v1.3 (FINAL).pdf p.3 |

40 questions, 33 scored for recall (one question = 0.030); 39 distinct gold files: {'01_policies': 13, '02_procedures': 5, '03_ledgerline_manuals': 6, '04_service_definitions': 4, '05_master_data': 4, '08_release_notes': 2, '09_email_archive': 3, '10_meeting_notes': 1, '11_training': 1}


In [10]:
import numpy, torch
print("numpy", numpy.__version__, "| torch", torch.__version__, "| numpy bridge:", torch.ones(2).cpu().numpy())

numpy 2.4.6 | torch 2.6.0+cu124 | numpy bridge: [1. 1.]


In [7]:
# A3 · Part 2 · 2. The grid: 108 configurations, every one timed on this machine.
# One chunking at a time: chunk, embed into Chroma, run its 3 searches x 2 reranker settings on the 40 questions,
# free memory. Results are saved after every configuration, so re-running this cell resumes where it stopped.
import gc

import torch

PARSERS_G = ("plain", "structured")
CHUNKERS_G = ("fixed", "recursive", "structure")
SIZES_G = (200, 400, 800)
OVERLAP = 0.15
EMBEDS = ["bge-small"]          # add "bge-base" for a second embedding model (adds dense and hybrid configurations)
SEARCHES = [("bm25", None)] + [(mode, e) for mode in ("dense", "hybrid") for e in EMBEDS]
K = 10
INDEX_ROOT = pathlib.Path.home() / ".cache" / "a3-rag-index-v2"  # outside OneDrive: the indexes take several GB
INDEX_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = INDEX_ROOT / "grid_results.json"
grid = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
WARMUP = "How do I raise a purchase order?"   # one untimed search per configuration: loads the models first
print("HNSW:", rag.HNSW, "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

units_by_parser = {"structured": units, "plain": C.parse(DOCS, parser="plain")}


def open_index(p, m, size):
    """The chunking's index from disk if it was built completely before, else chunk + embed + Chroma."""
    path = INDEX_ROOT / C.chunking_id(p, m, size, OVERLAP)
    marker = path / "built.json"
    if marker.exists() and set(json.loads(marker.read_text())) >= set(EMBEDS):
        return rag.Index.load(path), "loaded"
    t0 = time.time()
    chunks = C.chunk(units_by_parser[p], m, size, OVERLAP)
    idx = rag.Index.build(chunks, EMBEDS, path)
    marker.write_text(json.dumps(EMBEDS))
    return idx, f"built in {time.time() - t0:.0f} s, {idx.build_stats}"


t_grid = time.time()
for p in PARSERS_G:
    for m in CHUNKERS_G:
        for size in SIZES_G:
            cfgs = [rag.Config(parser=p, chunker=m, size=size, overlap=OVERLAP, embed=e or EMBEDS[0], retrieval=mode,
                               rerank=rr, k=K) for mode, e in SEARCHES for rr in (False, True)]
            todo = [c for c in cfgs if c.id not in grid]
            if not todo:
                continue
            idx, how = open_index(p, m, size)
            print(f"{C.chunking_id(p, m, size, OVERLAP):28} {len(idx.chunks):7,} chunks, {how}")
            for cfg in todo:
                ret = rag.Retriever(idx, cfg)
                ret.search(WARMUP)
                res = E.retrieval_only(lambda q, role: ret.search(q, role=role, k=K), G, qs=SUBSET)
                s, lat_s = res["summary"]["overall"], res["summary"]["latency_s"]
                grid[cfg.id] = {**cfg.to_json(), "n_chunks": len(idx.chunks),
                                "recall@5": s["recall@5"], "recall@10": s["recall@10"], "mrr": s["mrr"],
                                "p50_ms": round(lat_s["p50"] * 1000, 1), "p95_ms": round(lat_s["p95"] * 1000, 1),
                                "by_type": {t: v["recall@5"] for t, v in res["summary"]["by_type"].items() if v.get("n")},
                                "ranks": {r["id"]: r["retrieval"]["rank"] for r in res["records"]}}
                RESULTS.write_text(json.dumps(grid))
                print(f"   {cfg.retrieval:6} {'+rerank' if cfg.rerank else '':7}  recall@5 {s['recall@5']:.3f}  "
                      f"recall@10 {s['recall@10']:.3f}  MRR {s['mrr']:.3f}  p95 {lat_s['p95'] * 1000:6.1f} ms")
            del idx
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
print(f"{len(grid)} configurations saved in {RESULTS}; this run took {(time.time() - t_grid) / 60:.1f} min")

HNSW: {'space': 'cosine', 'ef_search': 4000, 'ef_construction': 200, 'max_neighbors': 32} | device: NVIDIA GeForce RTX 3060


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

plain-fixed-200-o15           19,859 chunks, built in 48 s, {'bge-small': {'embed_s': 28.46, 'chroma_add_s': 11.06, 'n': 19859, 'dim': 384}}
   bm25            recall@5 0.727  recall@10 0.849  MRR 0.492  p95    0.3 ms


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

   bm25   +rerank  recall@5 0.727  recall@10 0.849  MRR 0.559  p95   77.3 ms
   dense           recall@5 0.485  recall@10 0.606  MRR 0.424  p95   19.6 ms
   dense  +rerank  recall@5 0.606  recall@10 0.636  MRR 0.475  p95   84.9 ms
   hybrid          recall@5 0.758  recall@10 0.788  MRR 0.548  p95   20.8 ms
   hybrid +rerank  recall@5 0.727  recall@10 0.788  MRR 0.535  p95   87.0 ms
plain-fixed-400-o15            9,970 chunks, built in 33 s, {'bge-small': {'embed_s': 21.29, 'chroma_add_s': 6.61, 'n': 9970, 'dim': 384}}
   bm25            recall@5 0.636  recall@10 0.788  MRR 0.546  p95    0.3 ms
   bm25   +rerank  recall@5 0.758  recall@10 0.849  MRR 0.572  p95  120.7 ms
   dense           recall@5 0.455  recall@10 0.606  MRR 0.341  p95   21.8 ms
   dense  +rerank  recall@5 0.636  recall@10 0.667  MRR 0.513  p95  137.8 ms
   hybrid          recall@5 0.576  recall@10 0.697  MRR 0.473  p95   16.8 ms
   hybrid +rerank  recall@5 0.727  recall@10 0.849  MRR 0.565  p95  136.0 ms
plain-fixed-80

In [8]:
# A3 · Part 2 · 3. Every configuration, best recall@5 first; then what each setting does on its own.
ROWS = sorted(grid.values(), key=lambda r: (-r["recall@5"], -r["mrr"], r["p95_ms"]))
lines = ["| # | Parser | Chunker | Size | Search | Reranker | Chunks | Recall@5 | Recall@10 | MRR | p50 ms | p95 ms |",
         "|---:|---|---|---:|---|---|---:|---:|---:|---:|---:|---:|"]
for n, r in enumerate(ROWS, 1):
    search = r["retrieval"] if r["retrieval"] == "bm25" else f"{r['retrieval']} ({r['embed']})"
    lines.append(f"| {n} | {r['parser']} | {r['chunker']} | {r['size']} | {search} | {'on' if r['rerank'] else 'off'} | "
                 f"{r['n_chunks']:,} | {r['recall@5']:.3f} | {r['recall@10']:.3f} | {r['mrr']:.3f} | "
                 f"{r['p50_ms']:.1f} | {r['p95_ms']:.1f} |")
show_table("\n".join(lines))

lines = ["| Setting | Value | Configurations | Mean recall@5 | Best recall@5 | Median p95 ms |",
         "|---|---|---:|---:|---:|---:|"]
for knob in ("parser", "chunker", "size", "retrieval", "rerank"):
    for v in sorted({r[knob] for r in ROWS}, key=str):
        rs = [r for r in ROWS if r[knob] == v]
        lines.append(f"| {knob} | {v} | {len(rs)} | {statistics.mean(r['recall@5'] for r in rs):.3f} | "
                     f"{max(r['recall@5'] for r in rs):.3f} | {statistics.median(r['p95_ms'] for r in rs):.1f} |")
show_table("\n".join(lines))

| # | Parser | Chunker | Size | Search | Reranker | Chunks | Recall@5 | Recall@10 | MRR | p50 ms | p95 ms |
|---:|---|---|---:|---|---|---:|---:|---:|---:|---:|---:|
| 1 | structured | structure | 400 | hybrid (bge-small) | on | 13,734 | 0.909 | 0.970 | 0.761 | 131.0 | 139.2 |
| 2 | structured | structure | 400 | bm25 | on | 13,734 | 0.909 | 0.939 | 0.731 | 115.5 | 120.2 |
| 3 | structured | structure | 200 | bm25 | on | 28,953 | 0.909 | 0.939 | 0.689 | 68.3 | 74.5 |
| 4 | structured | recursive | 200 | bm25 | on | 28,711 | 0.879 | 0.909 | 0.712 | 70.4 | 74.8 |
| 5 | plain | structure | 400 | bm25 | on | 9,671 | 0.879 | 0.879 | 0.631 | 114.3 | 121.5 |
| 6 | structured | fixed | 200 | bm25 | on | 27,811 | 0.849 | 0.909 | 0.684 | 71.6 | 77.1 |
| 7 | structured | recursive | 400 | hybrid (bge-small) | on | 14,040 | 0.849 | 0.939 | 0.660 | 131.1 | 139.6 |
| 8 | structured | recursive | 400 | bm25 | on | 14,040 | 0.849 | 0.909 | 0.645 | 116.5 | 121.8 |
| 9 | structured | fixed | 400 | bm25 | on | 13,973 | 0.849 | 0.909 | 0.606 | 116.4 | 122.9 |
| 10 | structured | fixed | 200 | hybrid (bge-small) | on | 27,811 | 0.818 | 0.939 | 0.708 | 87.0 | 97.6 |
| 11 | structured | structure | 200 | hybrid (bge-small) | on | 28,953 | 0.818 | 0.939 | 0.640 | 82.5 | 88.7 |
| 12 | structured | fixed | 400 | hybrid (bge-small) | on | 13,973 | 0.818 | 0.909 | 0.575 | 131.7 | 137.7 |
| 13 | plain | structure | 200 | bm25 | on | 19,634 | 0.788 | 0.849 | 0.618 | 56.0 | 66.1 |
| 14 | plain | recursive | 200 | bm25 | on | 19,621 | 0.788 | 0.818 | 0.616 | 56.0 | 67.9 |
| 15 | structured | fixed | 200 | bm25 | off | 27,811 | 0.788 | 0.879 | 0.569 | 0.2 | 0.3 |
| 16 | structured | fixed | 800 | bm25 | off | 7,223 | 0.788 | 0.939 | 0.557 | 0.2 | 0.2 |
| 17 | structured | recursive | 800 | bm25 | on | 7,243 | 0.788 | 0.849 | 0.552 | 119.4 | 127.3 |
| 18 | plain | structure | 400 | hybrid (bge-small) | on | 9,671 | 0.788 | 0.818 | 0.551 | 129.6 | 136.6 |
| 19 | structured | structure | 800 | bm25 | off | 6,853 | 0.788 | 0.909 | 0.546 | 0.2 | 0.2 |
| 20 | structured | recursive | 400 | bm25 | off | 14,040 | 0.788 | 0.818 | 0.527 | 0.2 | 0.3 |
| 21 | structured | fixed | 800 | bm25 | on | 7,223 | 0.788 | 0.849 | 0.516 | 118.7 | 125.3 |
| 22 | structured | recursive | 200 | hybrid (bge-small) | on | 28,711 | 0.758 | 0.849 | 0.710 | 85.8 | 94.2 |
| 23 | plain | recursive | 800 | bm25 | on | 5,345 | 0.758 | 0.788 | 0.594 | 117.8 | 123.7 |
| 24 | plain | structure | 800 | bm25 | on | 5,072 | 0.758 | 0.788 | 0.593 | 118.5 | 123.5 |
| 25 | plain | recursive | 400 | bm25 | on | 9,988 | 0.758 | 0.849 | 0.587 | 112.3 | 118.8 |
| 26 | plain | fixed | 400 | bm25 | on | 9,970 | 0.758 | 0.849 | 0.572 | 113.4 | 120.7 |
| 27 | structured | recursive | 800 | bm25 | off | 7,243 | 0.758 | 0.939 | 0.550 | 0.2 | 0.3 |
| 28 | structured | recursive | 200 | bm25 | off | 28,711 | 0.758 | 0.818 | 0.549 | 0.2 | 0.3 |
| 29 | plain | fixed | 200 | hybrid (bge-small) | off | 19,859 | 0.758 | 0.788 | 0.548 | 15.9 | 20.8 |
| 30 | structured | fixed | 400 | bm25 | off | 13,973 | 0.758 | 0.818 | 0.524 | 0.2 | 0.2 |
| 31 | structured | structure | 200 | bm25 | off | 28,953 | 0.758 | 0.909 | 0.520 | 0.2 | 0.3 |
| 32 | structured | structure | 200 | hybrid (bge-small) | off | 28,953 | 0.758 | 0.849 | 0.500 | 14.3 | 18.2 |
| 33 | plain | recursive | 200 | bm25 | off | 19,621 | 0.758 | 0.849 | 0.477 | 0.2 | 0.3 |
| 34 | plain | recursive | 200 | hybrid (bge-small) | off | 19,621 | 0.727 | 0.818 | 0.600 | 16.6 | 19.0 |
| 35 | plain | recursive | 200 | hybrid (bge-small) | on | 19,621 | 0.727 | 0.758 | 0.591 | 72.7 | 77.1 |
| 36 | plain | structure | 800 | hybrid (bge-small) | on | 5,072 | 0.727 | 0.758 | 0.578 | 134.8 | 143.8 |
| 37 | plain | fixed | 400 | hybrid (bge-small) | on | 9,970 | 0.727 | 0.849 | 0.565 | 128.6 | 136.0 |
| 38 | plain | fixed | 200 | bm25 | on | 19,859 | 0.727 | 0.849 | 0.559 | 61.9 | 77.3 |
| 39 | plain | fixed | 200 | hybrid (bge-small) | on | 19,859 | 0.727 | 0.788 | 0.535 | 76.9 | 87.0 |
| 40 | structured | recursive | 800 | hybrid (bge-small) | on | 7,243 | 0.727 | 0.818 | 0.532 | 136.5 | 144.4 |
| 41 | structured | structure | 800 | bm25 | on | 6,853 | 0.727 | 0.788 | 0.496 | 118.5 | 123.5 |
| 42 | plain | fixed | 200 | bm25 | off | 19,859 | 0.727 | 0.849 | 0.492 | 0.2 | 0.3 |
| 43 | structured | fixed | 800 | hybrid (bge-small) | on | 7,223 | 0.727 | 0.818 | 0.478 | 134.9 | 143.3 |
| 44 | plain | structure | 200 | hybrid (bge-small) | on | 19,634 | 0.697 | 0.818 | 0.584 | 75.1 | 81.0 |
| 45 | plain | fixed | 800 | bm25 | on | 5,238 | 0.697 | 0.818 | 0.579 | 122.0 | 126.4 |
| 46 | plain | recursive | 800 | hybrid (bge-small) | on | 5,345 | 0.697 | 0.758 | 0.577 | 132.5 | 138.2 |
| 47 | plain | structure | 200 | hybrid (bge-small) | off | 19,634 | 0.697 | 0.849 | 0.554 | 18.9 | 24.0 |
| 48 | plain | recursive | 400 | hybrid (bge-small) | on | 9,988 | 0.697 | 0.849 | 0.550 | 129.8 | 136.9 |
| 49 | structured | structure | 400 | bm25 | off | 13,734 | 0.697 | 0.909 | 0.541 | 0.2 | 0.4 |
| 50 | structured | fixed | 200 | hybrid (bge-small) | off | 27,811 | 0.697 | 0.849 | 0.520 | 16.9 | 20.9 |
| 51 | plain | structure | 400 | bm25 | off | 9,671 | 0.697 | 0.788 | 0.495 | 0.2 | 0.3 |
| 52 | structured | recursive | 200 | hybrid (bge-small) | off | 28,711 | 0.697 | 0.818 | 0.487 | 14.4 | 17.8 |
| 53 | structured | structure | 400 | hybrid (bge-small) | off | 13,734 | 0.697 | 0.788 | 0.478 | 16.1 | 21.6 |
| 54 | plain | fixed | 800 | hybrid (bge-small) | on | 5,238 | 0.667 | 0.818 | 0.565 | 134.7 | 138.4 |
| 55 | structured | recursive | 400 | dense (bge-small) | on | 14,040 | 0.667 | 0.697 | 0.555 | 130.6 | 136.4 |
| 56 | plain | recursive | 200 | dense (bge-small) | on | 19,621 | 0.667 | 0.667 | 0.543 | 71.6 | 79.6 |
| 57 | structured | fixed | 400 | dense (bge-small) | on | 13,973 | 0.667 | 0.727 | 0.519 | 130.5 | 137.4 |
| 58 | plain | fixed | 800 | bm25 | off | 5,238 | 0.667 | 0.818 | 0.496 | 0.2 | 0.4 |
| 59 | structured | structure | 800 | hybrid (bge-small) | on | 6,853 | 0.667 | 0.788 | 0.483 | 135.8 | 142.9 |
| 60 | structured | fixed | 800 | dense (bge-small) | on | 7,223 | 0.667 | 0.667 | 0.452 | 133.3 | 140.1 |
| 61 | plain | structure | 200 | bm25 | off | 19,634 | 0.667 | 0.818 | 0.442 | 0.2 | 0.3 |
| 62 | structured | structure | 400 | dense (bge-small) | on | 13,734 | 0.636 | 0.667 | 0.551 | 128.7 | 136.4 |
| 63 | plain | fixed | 400 | bm25 | off | 9,970 | 0.636 | 0.788 | 0.546 | 0.2 | 0.3 |
| 64 | plain | fixed | 400 | dense (bge-small) | on | 9,970 | 0.636 | 0.667 | 0.513 | 128.9 | 137.8 |
| 65 | plain | recursive | 400 | bm25 | off | 9,988 | 0.636 | 0.818 | 0.496 | 0.2 | 0.3 |
| 66 | plain | structure | 800 | bm25 | off | 5,072 | 0.636 | 0.788 | 0.494 | 0.2 | 0.3 |
| 67 | plain | recursive | 800 | bm25 | off | 5,345 | 0.636 | 0.758 | 0.486 | 0.2 | 0.3 |
| 68 | structured | fixed | 400 | hybrid (bge-small) | off | 13,973 | 0.636 | 0.788 | 0.486 | 16.9 | 17.9 |
| 69 | structured | recursive | 400 | hybrid (bge-small) | off | 14,040 | 0.636 | 0.818 | 0.483 | 13.7 | 18.2 |
| 70 | structured | fixed | 800 | hybrid (bge-small) | off | 7,223 | 0.636 | 0.849 | 0.471 | 14.0 | 18.1 |
| 71 | plain | recursive | 400 | hybrid (bge-small) | off | 9,988 | 0.606 | 0.697 | 0.514 | 15.8 | 21.7 |
| 72 | plain | structure | 200 | dense (bge-small) | on | 19,634 | 0.606 | 0.636 | 0.476 | 74.1 | 82.8 |
| 73 | plain | fixed | 200 | dense (bge-small) | on | 19,859 | 0.606 | 0.636 | 0.475 | 77.5 | 84.9 |
| 74 | structured | recursive | 800 | dense (bge-small) | on | 7,243 | 0.606 | 0.606 | 0.446 | 135.0 | 141.1 |
| 75 | structured | recursive | 200 | dense (bge-small) | on | 28,711 | 0.576 | 0.606 | 0.566 | 84.2 | 92.9 |
| 76 | plain | recursive | 400 | dense (bge-small) | on | 9,988 | 0.576 | 0.667 | 0.524 | 130.8 | 136.2 |
| 77 | structured | structure | 200 | dense (bge-small) | on | 28,953 | 0.576 | 0.697 | 0.487 | 81.8 | 88.6 |
| 78 | plain | fixed | 400 | hybrid (bge-small) | off | 9,970 | 0.576 | 0.697 | 0.473 | 13.4 | 16.8 |
| 79 | structured | recursive | 800 | hybrid (bge-small) | off | 7,243 | 0.576 | 0.818 | 0.472 | 19.2 | 22.3 |
| 80 | structured | structure | 800 | hybrid (bge-small) | off | 6,853 | 0.576 | 0.818 | 0.436 | 13.1 | 15.2 |
| 81 | plain | structure | 400 | hybrid (bge-small) | off | 9,671 | 0.576 | 0.727 | 0.419 | 15.9 | 20.8 |
| 82 | structured | fixed | 200 | dense (bge-small) | on | 27,811 | 0.545 | 0.576 | 0.499 | 88.4 | 98.4 |
| 83 | plain | recursive | 200 | dense (bge-small) | off | 19,621 | 0.545 | 0.667 | 0.461 | 15.3 | 18.3 |
| 84 | structured | structure | 800 | dense (bge-small) | on | 6,853 | 0.545 | 0.576 | 0.437 | 133.8 | 143.9 |
| 85 | plain | fixed | 800 | dense (bge-small) | on | 5,238 | 0.545 | 0.576 | 0.433 | 135.9 | 142.1 |
| 86 | structured | structure | 800 | dense (bge-small) | off | 6,853 | 0.545 | 0.576 | 0.429 | 15.7 | 17.2 |
| 87 | plain | fixed | 800 | hybrid (bge-small) | off | 5,238 | 0.545 | 0.727 | 0.379 | 14.2 | 23.0 |
| 88 | structured | structure | 200 | dense (bge-small) | off | 28,953 | 0.515 | 0.667 | 0.423 | 14.3 | 18.0 |
| 89 | structured | recursive | 400 | dense (bge-small) | off | 14,040 | 0.515 | 0.636 | 0.402 | 15.8 | 18.8 |
| 90 | plain | recursive | 400 | dense (bge-small) | off | 9,988 | 0.515 | 0.667 | 0.392 | 13.9 | 17.8 |
| 91 | plain | structure | 800 | hybrid (bge-small) | off | 5,072 | 0.515 | 0.697 | 0.348 | 18.7 | 21.4 |
| 92 | plain | structure | 200 | dense (bge-small) | off | 19,634 | 0.485 | 0.606 | 0.448 | 17.4 | 22.5 |
| 93 | structured | recursive | 200 | dense (bge-small) | off | 28,711 | 0.485 | 0.545 | 0.439 | 15.4 | 20.9 |
| 94 | plain | fixed | 200 | dense (bge-small) | off | 19,859 | 0.485 | 0.606 | 0.424 | 16.2 | 19.6 |
| 95 | structured | fixed | 400 | dense (bge-small) | off | 13,973 | 0.485 | 0.576 | 0.382 | 14.0 | 18.7 |
| 96 | plain | recursive | 800 | hybrid (bge-small) | off | 5,345 | 0.485 | 0.667 | 0.319 | 13.3 | 17.2 |
| 97 | plain | structure | 400 | dense (bge-small) | on | 9,671 | 0.455 | 0.545 | 0.399 | 131.6 | 139.9 |
| 98 | structured | fixed | 200 | dense (bge-small) | off | 27,811 | 0.455 | 0.515 | 0.398 | 20.2 | 24.0 |
| 99 | structured | structure | 400 | dense (bge-small) | off | 13,734 | 0.455 | 0.636 | 0.390 | 14.2 | 18.4 |
| 100 | structured | recursive | 800 | dense (bge-small) | off | 7,243 | 0.455 | 0.485 | 0.377 | 14.4 | 20.8 |
| 101 | plain | fixed | 400 | dense (bge-small) | off | 9,970 | 0.455 | 0.606 | 0.341 | 14.2 | 21.8 |
| 102 | plain | structure | 800 | dense (bge-small) | on | 5,072 | 0.424 | 0.485 | 0.358 | 133.2 | 141.5 |
| 103 | plain | structure | 400 | dense (bge-small) | off | 9,671 | 0.394 | 0.515 | 0.353 | 14.1 | 22.1 |
| 104 | plain | recursive | 800 | dense (bge-small) | on | 5,345 | 0.394 | 0.455 | 0.326 | 131.2 | 138.0 |
| 105 | plain | fixed | 800 | dense (bge-small) | off | 5,238 | 0.394 | 0.485 | 0.279 | 15.3 | 19.8 |
| 106 | plain | structure | 800 | dense (bge-small) | off | 5,072 | 0.333 | 0.424 | 0.259 | 12.4 | 20.2 |
| 107 | structured | fixed | 800 | dense (bge-small) | off | 7,223 | 0.303 | 0.485 | 0.310 | 14.2 | 16.5 |
| 108 | plain | recursive | 800 | dense (bge-small) | off | 5,345 | 0.303 | 0.394 | 0.252 | 13.8 | 18.4 |

| Setting | Value | Configurations | Mean recall@5 | Best recall@5 | Median p95 ms |
|---|---|---:|---:|---:|---:|
| parser | plain | 54 | 0.625 | 0.879 | 45.0 |
| parser | structured | 54 | 0.687 | 0.909 | 49.2 |
| chunker | fixed | 36 | 0.656 | 0.849 | 50.5 |
| chunker | recursive | 36 | 0.657 | 0.879 | 45.1 |
| chunker | structure | 36 | 0.656 | 0.909 | 45.0 |
| size | 200 | 36 | 0.689 | 0.909 | 45.0 |
| size | 400 | 36 | 0.672 | 0.909 | 70.5 |
| size | 800 | 36 | 0.607 | 0.788 | 73.2 |
| retrieval | bm25 | 36 | 0.761 | 0.909 | 33.2 |
| retrieval | dense | 36 | 0.514 | 0.667 | 51.8 |
| retrieval | hybrid | 36 | 0.693 | 0.909 | 50.5 |
| rerank | False | 54 | 0.601 | 0.788 | 18.1 |
| rerank | True | 54 | 0.711 | 0.909 | 124.5 |

In [9]:
# A3 · Part 2 · 4. Pick by the rule fixed before the grid ran: the highest recall@5 among configurations whose
# search p95 is at most 200 ms on this machine; an exact tie goes to the higher MRR, then the lower p95.
LIMIT_MS = 200
eligible = [r for r in ROWS if r["p95_ms"] <= LIMIT_MS]
WINNER = max(eligible, key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
best_any = max(ROWS, key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
print(f"{len(eligible)} of {len(ROWS)} configurations have search p95 <= {LIMIT_MS} ms")
print("WINNER:", WINNER["id"])
print(f"  recall@5 {WINNER['recall@5']:.3f} (target >= 0.90: {'met' if WINNER['recall@5'] >= 0.90 else 'missed'}) | "
      f"recall@10 {WINNER['recall@10']:.3f} | MRR {WINNER['mrr']:.3f} | search p50 {WINNER['p50_ms']} ms, "
      f"p95 {WINNER['p95_ms']} ms (target <= {LIMIT_MS} ms: {'met' if WINNER['p95_ms'] <= LIMIT_MS else 'missed'})")
if best_any["id"] != WINNER["id"]:
    print(f"Best recall@5 at any speed: {best_any['id']} (recall@5 {best_any['recall@5']:.3f}, p95 {best_any['p95_ms']} ms)")

lines = ["| Type | Scored questions | Winner's recall@5 |", "|---|---:|---:|"]
for t, v in sorted(WINNER["by_type"].items(), key=lambda x: x[1]):
    lines.append(f"| {t} | {sum(1 for q in SCORED if q['type'] == t)} | {v:.2f} |")
show_table("\n".join(lines))
missed = [qid for qid, rank in WINNER["ranks"].items() if G.expected(G.by_id[qid]) == "answer" and (rank is None or rank > 5)]
for qid in missed:
    print(f"  missed at 5: {qid:7} rank {WINNER['ranks'][qid]}  {G.by_id[qid]['question']}")

108 of 108 configurations have search p95 <= 200 ms
WINNER: structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10
  recall@5 0.909 (target >= 0.90: met) | recall@10 0.970 | MRR 0.761 | search p50 131.0 ms, p95 139.2 ms (target <= 200 ms: met)


| Type | Scored questions | Winner's recall@5 |
|---|---:|---:|
| procedure | 4 | 0.75 |
| table_lookup | 4 | 0.75 |
| version_conflict | 4 | 0.75 |
| single_lookup | 4 | 1.00 |
| rule | 4 | 1.00 |
| service_field | 4 | 1.00 |
| multi_hop | 4 | 1.00 |
| exact_code | 4 | 1.00 |
| permission | 1 | 1.00 |

  missed at 5: Q-110   rank 9  How do I raise a purchase request?
  missed at 5: Q-422   rank None  What is the warehouse WH-MAN-BAT?
  missed at 5: Q-615   rank 6  Did chargers always need UN 38.3?


In [10]:
# A3 · Part 2 · 5. Exact codes: an exact-code question where dense search misses the gold source and BM25 finds it.
# Both on the winner's chunking, without the reranker, so each search is seen on its own.
wcfg = rag.Config.from_id(WINNER["id"])
widx = rag.Index.load(INDEX_ROOT / wcfg.chunking)
plain_search = {name: rag.Retriever(widx, rag.Config(**{**wcfg.__dict__, "retrieval": name, "rerank": False}))
                for name in ("dense", "bm25")}


def is_gold(q, it):
    return any(E.loc_match(G, g, it["file"], it["loc"]) for g in G.gold(q))


def first_gold(q, items):
    return next((it["rank"] for it in items if is_gold(q, it)), None)


found = None
for q in [q for q in SUBSET if q["type"] == "exact_code"]:
    res = {name: r.search(q["question"], role=C.role_of(q), k=K) for name, r in plain_search.items()}
    ranks = {name: first_gold(q, out["items"]) for name, out in res.items()}
    print(f"{q['id']:7} dense: gold at rank {ranks['dense']}, BM25: gold at rank {ranks['bm25']}   {q['question']}")
    if found is None and ranks["bm25"] and ranks["bm25"] <= 3 and (ranks["dense"] is None or ranks["dense"] > 5):
        found = (q, res)

if found is None:
    print("\nNo exact-code question in the subset where dense misses and BM25 finds the gold source in its top 3.")
else:
    q, res = found
    print(f"\n{q['id']}: {q['question']}\ngold: {[gold_label(g) for g in G.gold(q)]}")
    for name in ("dense", "bm25"):
        print(f"--- {name}, top 3")
        for it in res[name]["items"][:3]:
            print(f"  {it['rank']}. {'GOLD' if is_gold(q, it) else '    '} {it['cite']}")

Q-0412  dense: gold at rank 3, BM25: gold at rank 1   What does ERR-PO-0412 mean and how do I fix it?
Q-705   dense: gold at rank None, BM25: gold at rank 6   What does ERR-SUP-0118 mean?
Q-703   dense: gold at rank None, BM25: gold at rank 7   What does ERR-PO-0610 mean?
Q-711   dense: gold at rank None, BM25: gold at rank 2   What does ERR-SEC-0001 say?

Q-711: What does ERR-SEC-0001 say?
gold: ['MAN-ERR-72 Message and Error Code Reference v7.2.3.pdf p.23']
--- dense, top 3
  1.      MAN-SAL-72 v7.2.3, p.7, § Questions users ask
  2.      MAN-SAL-72 v7.2.3, pp.36-37, § Questions users ask
  3.      MAN-AP-72 v7.2.3, p.64, § ERR-TAX-0001
--- bm25, top 3
  1.      Ledgerline_7.2_tables_and_fields.xlsx, Error codes rows 209-227
  2. GOLD MAN-ERR-72 v7.2.3, p.23, § ERR-SEC-0001 ACCESS DENIED
  3.      svc:po_approve, § request.fields, service po_approve


### A3 · Part 2 · 6. Sensitivity check: BM25's k1 and b

The grid used BM25's textbook settings everywhere: **k1 = 1.2** (how fast repeated words stop counting) and
**b = 0.75** (how strongly long chunks are scaled down). This check, run after the grid, varies both on the winner's
chunking: k1 ∈ {0.6, 0.9, 1.2, 1.5, 2.0} × b ∈ {0.3, 0.5, 0.75, 0.9, 1.0}, for BM25 on its own and for the full
winning pipeline (hybrid + reranker).

**Decision rule, fixed before this cell ran:** keep k1 = 1.2, b = 0.75 unless a setting raises the winning pipeline's
recall@5 by **at least two questions** (2 / 33 = 0.061) with search p95 still ≤ 200 ms. On 33 questions, one question
of difference is within noise.

In [11]:
# A3 · Part 2 · 6. Sensitivity check, run after the main grid: BM25's k1 and b on the winner's chunking.
K1S = (0.6, 0.9, 1.2, 1.5, 2.0)
BS = (0.3, 0.5, 0.75, 0.9, 1.0)
wcfg = rag.Config.from_id(WINNER["id"])
tidx = rag.Index.load(INDEX_ROOT / wcfg.chunking)
texts = [c.text for c in tidx.chunks]
pipelines = {"BM25 alone": rag.Config(**{**wcfg.__dict__, "retrieval": "bm25", "rerank": False}),
             "winner (hybrid + reranker)": wcfg}
t0 = time.time()
tune = []
for k1 in K1S:
    for b in BS:
        tidx._bm25 = rag.BM25(texts, k1=k1, b=b)          # the index's BM25, rebuilt with these settings
        for name, cfg in pipelines.items():
            ret = rag.Retriever(tidx, cfg)
            ret.search(WARMUP)
            res = E.retrieval_only(lambda q, role: ret.search(q, role=role, k=K), G, qs=SUBSET)
            s, lat_s = res["summary"]["overall"], res["summary"]["latency_s"]
            tune.append({"pipeline": name, "k1": k1, "b": b, "recall@5": s["recall@5"], "recall@10": s["recall@10"],
                         "mrr": s["mrr"], "p95_ms": round(lat_s["p95"] * 1000, 1),
                         "ranks": {r["id"]: r["retrieval"]["rank"] for r in res["records"]}})
tidx._bm25 = None
(INDEX_ROOT / "bm25_tuning.json").write_text(json.dumps(tune))
print(f"{len(tune)} runs in {(time.time() - t0) / 60:.1f} min\n")

for name in pipelines:
    rs = {(r["k1"], r["b"]): r for r in tune if r["pipeline"] == name}
    lines = [f"**{name}**: recall@5 (MRR) for each k1 (rows) and b (columns); the default is k1 = 1.2, b = 0.75", "",
             "| k1 \\ b | " + " | ".join(str(b) for b in BS) + " |", "|---:|" + "---:|" * len(BS)]
    for k1 in K1S:
        lines.append(f"| {k1} | " + " | ".join(f"{rs[k1, b]['recall@5']:.3f} ({rs[k1, b]['mrr']:.3f})" for b in BS) + " |")
    show_table("\n".join(lines))
    base = rs[1.2, 0.75]
    best = max(rs.values(), key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
    print(f"{name}: default (k1 1.2, b 0.75) recall@5 {base['recall@5']:.3f}, MRR {base['mrr']:.3f} | best (k1 {best['k1']}, "
          f"b {best['b']}) recall@5 {best['recall@5']:.3f}, MRR {best['mrr']:.3f}, p95 {best['p95_ms']} ms | "
          f"{round((best['recall@5'] - base['recall@5']) * len(SCORED)):+d} questions")
    moved = {qid: (base["ranks"][qid], best["ranks"][qid]) for qid in base["ranks"] if base["ranks"][qid] != best["ranks"][qid]}
    print("   gold rank, default -> best:", moved or "no change", "\n")

w = [r for r in tune if r["pipeline"] == "winner (hybrid + reranker)"]
base_w = next(r for r in w if r["k1"] == 1.2 and r["b"] == 0.75)
best_w = max((r for r in w if r["p95_ms"] <= LIMIT_MS), key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
switch = round((best_w["recall@5"] - base_w["recall@5"]) * len(SCORED)) >= 2
print("Default run reproduces the grid's winner:", base_w["recall@5"] == WINNER["recall@5"],
      f"({base_w['recall@5']:.3f} vs {WINNER['recall@5']:.3f})")
print("DECISION:", f"switch to k1 = {best_w['k1']}, b = {best_w['b']}" if switch
      else "keep k1 = 1.2, b = 0.75 (no setting is at least two questions better)")

50 runs in 3.4 min



**BM25 alone**: recall@5 (MRR) for each k1 (rows) and b (columns); the default is k1 = 1.2, b = 0.75

| k1 \ b | 0.3 | 0.5 | 0.75 | 0.9 | 1.0 |
|---:|---:|---:|---:|---:|---:|
| 0.6 | 0.788 (0.583) | 0.788 (0.581) | 0.788 (0.554) | 0.727 (0.530) | 0.727 (0.540) |
| 0.9 | 0.849 (0.570) | 0.788 (0.582) | 0.727 (0.543) | 0.667 (0.530) | 0.667 (0.543) |
| 1.2 | 0.818 (0.586) | 0.758 (0.588) | 0.697 (0.541) | 0.667 (0.548) | 0.667 (0.544) |
| 1.5 | 0.818 (0.591) | 0.788 (0.604) | 0.697 (0.556) | 0.667 (0.566) | 0.667 (0.564) |
| 2.0 | 0.818 (0.606) | 0.788 (0.616) | 0.697 (0.571) | 0.667 (0.572) | 0.667 (0.562) |

BM25 alone: default (k1 1.2, b 0.75) recall@5 0.697, MRR 0.541 | best (k1 0.9, b 0.3) recall@5 0.849, MRR 0.570, p95 0.2 ms | +5 questions
   gold rank, default -> best: {'Q-705': (6, 5), 'Q-502': (3, 2), 'Q-104': (8, 7), 'Q-112': (1, 2), 'Q-208': (6, 5), 'Q-322': (2, 3), 'Q-002': (3, 1), 'Q-020': (4, 3), 'Q-413': (10, 2), 'Q-618': (None, 10), 'Q-615': (6, 5), 'Q-607': (6, 4)} 



**winner (hybrid + reranker)**: recall@5 (MRR) for each k1 (rows) and b (columns); the default is k1 = 1.2, b = 0.75

| k1 \ b | 0.3 | 0.5 | 0.75 | 0.9 | 1.0 |
|---:|---:|---:|---:|---:|---:|
| 0.6 | 0.909 (0.761) | 0.909 (0.761) | 0.909 (0.761) | 0.879 (0.736) | 0.909 (0.717) |
| 0.9 | 0.909 (0.761) | 0.909 (0.761) | 0.909 (0.761) | 0.909 (0.761) | 0.909 (0.717) |
| 1.2 | 0.909 (0.741) | 0.909 (0.741) | 0.909 (0.761) | 0.909 (0.736) | 0.909 (0.711) |
| 1.5 | 0.909 (0.741) | 0.909 (0.736) | 0.909 (0.736) | 0.939 (0.742) | 0.939 (0.718) |
| 2.0 | 0.879 (0.726) | 0.909 (0.741) | 0.939 (0.742) | 0.939 (0.742) | 0.909 (0.717) |

winner (hybrid + reranker): default (k1 1.2, b 0.75) recall@5 0.909, MRR 0.761 | best (k1 1.5, b 0.9) recall@5 0.939, MRR 0.742, p95 134.9 ms | +1 questions
   gold rank, default -> best: {'Q-322': (1, 3), 'Q-615': (6, 5)} 

Default run reproduces the grid's winner: True (0.909 vs 0.909)
DECISION: keep k1 = 1.2, b = 0.75 (no setting is at least two questions better)


## 8 · The shared answer step (recorded, replayed)

Every method in the race hands its passages to the same answer prompt: answer only from the numbered sources, cite
each fact, give the rule in force today, refuse confidential HR content. We rebuild the race's prompt for Emma's
question from the exact passages RAG retrieved in the race (`race_examples.json` lists their chunk ids), and replay
the recorded answer and judge calls.

In [ ]:
RX = json.loads((pathlib.Path(rag.__file__).parent / "race_examples.json").read_text())
from ledgerline.course.l3.wiki_llm import WikiLLM
ans_llm = WikiLLM("L3v4_race_rag_answers", max_tokens=2048)
judge_llm = WikiLLM("L3v4_race_rag_judge", max_tokens=512)
q = G.by_id["Q-EMMA"]
items = [by_id[i].as_item() for i in RX["questions"]["Q-EMMA"]["items"]]
msgs = E.answer_messages(q, items, answer_words=RX["answer_words"])
print(msgs[0]["content"])
print("…", len(msgs[1]["content"]), "characters of sources")

In [ ]:
a = E.answer(q, items, ans_llm, answer_words=RX["answer_words"])
print(a["answer"])
print(f"\nreplayed: {a['replayed']} · {a['tokens_in']} tokens in · {a['llm_seconds']} s when it was recorded")
print("exact-fact check:", E.fact_check(q, a["answer"], G))
print("judge:", E.judge(q, a["answer"], judge_llm, G))

Which of the question's 11 gold sources did RAG bring back? The cites tell you why the answer misses £104 against
£119 and PUR.CERT_CHK.

In [ ]:
for it in items:
    print(" ", it["cite"])
print("gold:", [g["file"].split("/")[-1] for g in G.gold(q)])

### TODO (exercise 3) · The same step for ERR-PO-0412
Rebuild the items for `Q-0412` from `RX`, replay the answer with `ans_llm`, and run the exact-fact check.

In [ ]:
if SOLUTION:
    q2 = G.by_id["Q-0412"]
    items2 = [by_id[i].as_item() for i in RX["questions"]["Q-0412"]["items"]]
    a2 = E.answer(q2, items2, ans_llm, answer_words=RX["answer_words"])
    print(a2["answer"][:900])
    print(E.fact_check(q2, a2["answer"], G))

## 9 · The LLM Wiki

Karpathy's LLM Wiki ("LLM Wiki", a gist, April 2026): an LLM reads the raw sources once and writes a wiki of linked
pages, which it then maintains. Three layers: the raw sources (never edited), the wiki (pages the LLM owns, an
`index.md` and a `log.md`) and the schema (`AGENTS.md`). Three operations: ingest, query, lint. Our wiki ships with
the course package.

In [ ]:
from ledgerline.course.l3 import wiki_query as WQ
w = WQ.Wiki(dump=DOCS)
print(len(w.pages), "pages")
print((w.root / "AGENTS.md").read_text()[:700])
print(w.page_text("sup-brightcell-energy")[:1600])

In [ ]:
print((w.root / "index.md").read_text()[:700])
print([l for l in (w.root / "log.md").read_text().splitlines() if l.startswith("## [")][:3])

**Search over the wiki's sections** (no LLM): BM25 and bge-small over every section of every page, fused.

In [ ]:
si = WQ.SectionIndex(w)
for hit in si.search("Can I buy the battery packs from Brightcell Energy? They quote £104.", role="PUR-EXEC", k=5):
    print(f"  [[{hit['page']}]] {hit['title']} > {hit['section']}")

**Query by navigation** (recorded, replayed): the model reads `index.md`, opens pages, follows links, and stops when
it has enough. Each step is a model call.

In [ ]:
nav_llm = WikiLLM("L3v4_wiki_nav", max_tokens=4000)
out = WQ.Navigator(w, nav_llm).navigate(q["question"], "PUR-EXEC")
for s in out["chain"]:
    print(" ", s)
print(len(out["items"]), "pages go to the shared answer step")

## 10 · The race, from the log

The full race ran on Modal: 224 questions, four methods, the same answer prompt (250 words), the same model and
judge. Its log is in the course repository at `lectures/L3-the-memory/experiments/logs/race.json`.

In [ ]:
import urllib.request
RACE_URL = "https://raw.githubusercontent.com/RajatDandekar/agents-for-enterprises/main/lectures/L3-the-memory/experiments/logs/race.json"
local = pathlib.Path("experiments/logs/race.json")   # when you run it inside the course repository
race = json.loads(local.read_text()) if local.exists() else json.loads(urllib.request.urlopen(RACE_URL).read())
print(f"{'method':10} {'judge':>6} {'facts':>6} {'cites':>6} {'search p95':>11} {'answer p50':>11} {'$/q':>8}")
for m, o in race["overall"].items():
    print(f"{m:10} {o['answer_correct']:6.1%} {o['fact_correct']:6.1%} {o['citation_correct']:6.1%} "
          f"{o['retrieval_p95_s'] * 1000:9.0f}ms {o['answer_p50_s']:10.1f}s {o['cost_per_q_usd']:8.4f}")
print("winner:", race["decision"]["winner"])

### TODO (exercise 4) · Your own question
Write a question Emma might ask, search it with `best` (RAG) and with `si` (the wiki's sections), and compare the
top 3 of each. Which one would you trust, and why?

In [ ]:
MY_Q = "Who must approve a purchase order of £300,000?"
if SOLUTION:
    for it in best.search(MY_Q, role="PUR-EXEC")["items"][:3]:
        print("  RAG ", it["cite"])
    for hit in si.search(MY_Q, role="PUR-EXEC", k=3):
        print("  wiki", f"[[{hit['page']}]] {hit['section']}")

## A3 · Part 3 · A mini LLM Wiki

My wiki covers `01_policies` and `02_procedures` (50 files). Its schema is my own `wiki/AGENTS.md`, which has 11 page
types, slug rules, the sections for each type, the citation format, and the rules for versions and permissions. The
course's wiki code (`wiki_build`) does the ingest, using my schema in place of the course's. The model is Gemini 3.8
Flash, the same one the course uses. Every model call is recorded in `cassettes/A3_wiki_ingest.jsonl`, so a rerun
replays it for free and the notebook can say which numbers are live and which are replayed.

Order: a 3-file pilot to measure cost, then the full ingest only if the pilot's estimate fits the budget.

In [4]:
# A3 · Part 3 · 0. Setup: my schema, my scope, a spending cap, and fixes so the course's wiki code runs on
# Windows exactly as it does on Linux. Nothing here calls a model.
import collections
import getpass
import json
import os
import pathlib
import sys
import time

import ledgerline.llm as LL
from ledgerline.course.l3 import wiki_build as WB
from ledgerline.course.l3 import wiki_llm as WL
from ledgerline.course.l3 import wiki_sources as WS

A3 = pathlib.Path(DOCS).absolute().parent          # Assignment_A3 (the dump was cloned into it)
WIKI = A3 / "wiki"

# 1) Paths. wiki_sources lists files with os.path.relpath, like corpus did (fixed in "A3 · Running locally"). With
#    back-slashes, "01_policies/HR/CONFIDENTIAL/" never matches, so the HR files would be treated as staff files
#    and their facts would land on staff pages. The "/old/" check for superseded files would fail too.
if os.name == "nt" and not getattr(WS.all_files, "posix_paths", False):
    _all_files = WS.all_files

    def all_files(root=WS.DUMP):
        return sorted(f.replace("\\", "/") for f in _all_files(root))

    all_files.posix_paths = True
    WS.all_files = all_files

# 2) Text encoding. The wiki code reads and writes pages with Path.read_text()/write_text() and no encoding. On
#    Windows that means cp1252, which cannot write characters such as the arrow or the less-than-or-equal sign that
#    the model often uses, and which misreads AGENTS.md. Default every text read and write to UTF-8, as on Linux.
if not sys.flags.utf8_mode and not getattr(pathlib.Path.read_text, "utf8", False):
    _read_text, _write_text, _open = pathlib.Path.read_text, pathlib.Path.write_text, pathlib.Path.open

    def read_text(self, encoding=None, *args, **kwargs):
        return _read_text(self, encoding or "utf-8", *args, **kwargs)

    def write_text(self, data, encoding=None, *args, **kwargs):
        return _write_text(self, data, encoding or "utf-8", *args, **kwargs)

    def path_open(self, mode="r", buffering=-1, encoding=None, errors=None, newline=None):
        if "b" not in mode and encoding is None:
            encoding = "utf-8"
        return _open(self, mode, buffering, encoding, errors, newline)

    read_text.utf8 = True
    pathlib.Path.read_text, pathlib.Path.write_text, pathlib.Path.open = read_text, write_text, path_open

# 3) My schema and my page types. The course code builds a registry of known slugs from the whole dump. It includes
#    manuals, release notes, meetings, service groups and cost centres, which are outside my scope and are not types
#    in my schema, so keep only registry entries of my types.
WB.SCHEMA = (WIKI / "AGENTS.md").read_text(encoding="utf-8")
MY_TYPES = ["policy", "procedure", "screen", "error", "supplier", "item-family", "role", "person", "data-source",
            "topic", "answer"]
WB.TYPES = MY_TYPES
if not getattr(WS.build_registry, "my_types", False):
    _build_registry = WS.build_registry

    def build_registry(srcs):
        return {s: r for s, r in _build_registry(srcs).items() if s.startswith("_") or r["type"] in MY_TYPES}

    build_registry.my_types = True
    WS.build_registry = build_registry

# 4) The scope: 01_policies + 02_procedures. Check the paths and permissions before any money is spent.
SCOPE = [f for f in WS.all_files(str(DOCS)) if f.startswith(("01_policies/", "02_procedures/"))]
SCOPE_SRCS = {s.file: s for s in WS.load_sources(str(DOCS), SCOPE)}
hr_files = sorted(f for f, s in SCOPE_SRCS.items() if s.audience == "hr")
assert len(SCOPE) == 50, f"expected 50 files in scope, found {len(SCOPE)}"
assert len(hr_files) == 4, f"expected the 4 HR files to be flagged hr, found {hr_files}"
print(f"scope: {len(SCOPE)} files, {sum(len(s.text) for s in SCOPE_SRCS.values()):,} characters;",
      "status", dict(collections.Counter(s.status for s in SCOPE_SRCS.values())), "| hr files:", len(hr_files))
print(f"schema: wiki/AGENTS.md, {len(WB.SCHEMA):,} characters | page types: {', '.join(MY_TYPES)}")

# 5) Where model calls are recorded, where they go, and the key. The key is typed in, so it never sits in the notebook.
LL.CASSETTE_DIR = A3 / "cassettes"
if not (os.environ.get("GEMINI_API_KEY") or os.environ.get("OPENROUTER_API_KEY")):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY (leave empty to replay only): ").strip()
if os.environ.get("GEMINI_API_KEY") and not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["LEDGER_WIKI_PROVIDER"] = "google-direct"   # the same model through Google's own endpoint
print("cassettes:", LL.CASSETTE_DIR, "| live calls go to:", WL.provider(),
      "| key:", "set" if (os.environ.get("GEMINI_API_KEY") or os.environ.get("OPENROUTER_API_KEY")) else "none (replay only)")


# 6) A hard spending cap. Every live call is costed at the course's price ($0.825 / $4.125 per million tokens). Once
#    the live spend passes the cap, further calls fail instead of being sent; the build logs those as errors.
class CappedLLM(WL.WikiLLM):
    def __init__(self, cassette, cap_usd, **kw):
        super().__init__(cassette, **kw)
        self.cap_usd = cap_usd

    def live_usd(self):
        return sum(WL.reply_usd(r) for r in self.log if not r.replayed)

    def _live(self, messages, tools, response_format, **kw):
        if self.live_usd() >= self.cap_usd:
            raise RuntimeError(f"spending cap reached: ${self.live_usd():.3f} of ${self.cap_usd:.2f}")
        return super()._live(messages, tools, response_format, **kw)


def spend(llm):
    live = [r for r in llm.log if not r.replayed]
    return {"calls": len(llm.log), "live_calls": len(live), "replayed_calls": len(llm.log) - len(live),
            "live_usd": round(sum(WL.reply_usd(r) for r in live), 4)}

scope: 50 files, 438,340 characters; status {'future': 2, 'current': 41, 'duplicate-file': 1, 'superseded': 6} | hr files: 4
schema: wiki/AGENTS.md, 8,217 characters | page types: policy, procedure, screen, error, supplier, item-family, role, person, data-source, topic, answer
cassettes: C:\Users\harma\OneDrive\Desktop\python\vizuara\Enterprise_Agents\Assignment_A3\cassettes | live calls go to: google-direct | key: set


In [9]:
# A3 · Part 3 · 1. Pilot: ingest 3 of the 50 files into a scratch folder (not the submitted wiki) and measure the cost.
# The three cover a policy that replaced an older version, a procedure, and a confidential HR file.
# The pilot's READ calls are recorded, and the full ingest replays them for free: the same file gives the same prompt.
PILOT = ["01_policies/POL-PUR-001_Purchase_Policy_v4.1.pdf",
         "02_procedures/SOP-PUR-002 PR to PO v1.4.pdf",
         "01_policies/HR/CONFIDENTIAL/HR-CONF-007 Salary Bands 2026-27.pdf"]
PILOT_DIR = pathlib.Path.home() / ".cache" / "a3-wiki-pilot"
pilot_llm = CappedLLM("A3_wiki_ingest", cap_usd=0.40)
t0 = time.time()
pilot = WB.build(str(DOCS), PILOT_DIR, pilot_llm, workers=8, files=PILOT)
pilot_s = spend(pilot_llm)
print(f"\n{pilot['pages']} pages {pilot['pages_by_type']} | {pilot['links']} links ({pilot['links_resolved']} resolve) | "
      f"{pilot['references']} references | read errors {pilot['read_errors']} | {time.time() - t0:.0f} s")
print("spend:", pilot_s)

# Rough estimate for all 50 files: scale the pilot's cost by characters. It is rough because the number of pages
# written does not grow exactly with the text.
chars = lambda fs: sum(len(SCOPE_SRCS[f].text) for f in fs)
ratio = chars(SCOPE) / chars(PILOT)
print(f"the 50 files have {ratio:.1f}x the pilot's characters -> estimated full ingest ${pilot['llm']['all']['usd'] * ratio:.2f}")

# Did the schema hold? The HR page must sit in restricted/ and appear in no staff index; no page of a type
# outside my schema.
idx = (PILOT_DIR / "index.md").read_text()
man = json.loads((PILOT_DIR / "manifest.json").read_text())
print("hr pages:", sorted(s for s, p in man["pages"].items() if p["audience"] == "hr"),
      "| HR slug in staff index.md:", any(s in idx for s, p in man["pages"].items() if p["audience"] == "hr"))
print("pages:", sorted(man["pages"]))

parsed 3 sources in 35.6s; registry 319 slugs
READ: 3 sources, 3 parts
READ done in 28s {'calls': 3, 'tokens_in': 19665, 'tokens_out': 8557, 'llm_seconds': 56.3, 'usd': 0.0515, 'replayed': 0}
WRITE done in 14s {'calls': 14, 'tokens_in': 45817, 'tokens_out': 9370, 'llm_seconds': 90.8, 'usd': 0.0765, 'replayed': 0}

14 pages {'error': 2, 'policy': 3, 'item-family': 1, 'screen': 4, 'procedure': 1, 'topic': 1, 'person': 2} | 41 links (41 resolve) | 62 references | read errors 0 | 82 s
spend: {'calls': 18, 'live_calls': 18, 'replayed_calls': 0, 'live_usd': 0.1288}
the 50 files have 19.0x the pilot's characters -> estimated full ingest $2.44
hr pages: ['hr-conf-007'] | HR slug in staff index.md: False
pages: ['err-po-0412', 'err-scr-0221', 'hr-conf-007', 'items-bat', 'pol-apr-002', 'pol-pur-001', 'screen-fin-cc-01', 'screen-pur-po-01', 'screen-pur-por-05', 'screen-pur-pr-01', 'sop-pur-002', 'topic-price-tolerance', 'user-dokafor', 'user-hweber']


In [10]:
# A3 · Part 3 · 2. The full ingest: all 50 files into wiki/. The pilot projected $2.1-2.4, so the cap is $2.40.
# The cap stops live calls if the spend runs past it (calls already recorded replay for free and do not count).
WIKI_CAP_USD = 2.40
llm = CappedLLM("A3_wiki_ingest", cap_usd=WIKI_CAP_USD)
t0 = time.time()
build = WB.build(str(DOCS), WIKI, llm, workers=8, files=SCOPE)
build["spend"] = spend(llm)
build["wall_seconds"] = round(time.time() - t0, 1)
(WIKI / "build_report.json").write_text(json.dumps({k: v for k, v in build.items() if k != "merge_map"}, indent=1))
print(f"{build['pages']} pages {build['pages_by_type']}")
print(f"{build['links']} links ({build['links_resolved']} resolve) | {build['references']} references | "
      f"{build['facts']} facts, {build['facts_quote_verified']} quotes verified | orphans {len(build['orphans'])} | "
      f"read errors {build['read_errors']}")
print("time (s):", build["seconds"], "| spend:", build["spend"], "| cost at the course price:", build["llm"]["all"])

parsed 50 sources in 32.8s; registry 319 slugs
READ: 50 sources, 50 parts
  read 50/50  {'calls': 50, 'tokens_in': 329026, 'tokens_out': 117929, 'llm_seconds': 783.2, 'usd': 0.7579, 'replayed': 3}
READ done in 99s {'calls': 50, 'tokens_in': 329026, 'tokens_out': 117929, 'llm_seconds': 783.2, 'usd': 0.7579, 'replayed': 3}
  wrote 50/118  {'calls': 101, 'tokens_in': 517691, 'tokens_out': 167235, 'llm_seconds': 1189.3, 'usd': 1.1169, 'replayed': 4}
  wrote 100/118  {'calls': 151, 'tokens_in': 686859, 'tokens_out': 207219, 'llm_seconds': 1545.2, 'usd': 1.4214, 'replayed': 4}
WRITE done in 110s {'calls': 118, 'tokens_in': 413901, 'tokens_out': 101110, 'llm_seconds': 870.9, 'usd': 0.7585, 'replayed': 1}
118 pages {'policy': 17, 'data-source': 1, 'error': 23, 'item-family': 2, 'role': 5, 'screen': 29, 'procedure': 23, 'supplier': 4, 'topic': 1, 'person': 13}
496 links (496 resolve) | 773 references | 838 facts, 830 quotes verified | orphans 4 | read errors 0
time (s): {'parse': 32.8, 'read': 

In [6]:
# A3 · Part 3 · 3. Query by navigation: three questions from my golden subset, one of them multi-hop.
# The model reads index.md, opens pages, follows [[links]] and, if it needs to, opens a raw source. A separate step
# answers from what it opened (the same answer prompt, model and 250-word cap the Part 4 race uses), and the answer is
# checked with the course's fact check, LLM judge and citation check. Every call is recorded in cassettes/.
from ledgerline.course.l3 import corpus as C
from ledgerline.course.l3 import evaluate as E
from ledgerline.course.l3 import wiki_query as WQ

G = G if "G" in globals() else E.Golden(DOCS)
LOGS = A3 / "logs"
LOGS.mkdir(exist_ok=True)
w = WQ.Wiki(WIKI, dump=DOCS)
nav_llm = CappedLLM("A3_wiki_nav", cap_usd=0.25, max_tokens=4000)
ans_llm = CappedLLM("A3_answers", cap_usd=0.10, max_tokens=2048)
judge_llm = CappedLLM("A3_judge", cap_usd=0.05, max_tokens=512)


def show_chain(chain):
    for s in chain:
        st = s["step"]
        if st == "index":
            print("    read index.md")
        elif st in ("page", "link", "list"):
            via = f"  (a link on [[{s['from']}]])" if s.get("from") else "  (from the index)"
            print(f"    turn {s['turn']}: open [[{s['what']}]]{via}")
        elif st == "source":
            print(f"    turn {s['turn']}: open raw source {s['what']} (page {s.get('page')}, find {s.get('find')!r}): "
                  f"{s.get('units')} part(s){' | ' + s['error'] if s.get('error') else ''}")
        elif st in ("denied", "missing"):
            print(f"    turn {s['turn']}: [[{s['what']}]] {st}")
        elif st == "done":
            print(f"    turn {s['turn']}: done ({s.get('why', '')})")


QUERIES = [("Q-510", "multi-hop: a direct-order limit that an amendment changes later"),
           ("Q-610", "two versions of one policy, one not yet in force"),
           ("Q-213", "one rule in one clause")]
query_log = []
for qid, why in QUERIES:
    q = G.by_id[qid]
    role = C.role_of(q)
    nav = WQ.Navigator(w, nav_llm).navigate(q["question"], role)
    a = E.answer(q, nav["items"], ans_llm, answer_words=E.RACE_WORDS)
    fact, verdict = E.fact_check(q, a["answer"], G), E.judge(q, a["answer"], judge_llm, G)
    cites = E.citation_check(q, a, nav["items"], G)
    print(f"=== {qid} ({q['type']}; {why}), asked as {role}\nQ: {q['question']}")
    print(f"  chain: {nav['nav_calls']} model calls, {nav['seconds']:.1f} s")
    show_chain(nav["chain"])
    print("  answer:\n    " + a["answer"].replace("\n", "\n    "))
    print("  sources the answer cites:")
    for n in a["cited"]:
        it = nav["items"][n - 1]
        print(f"    [{n}] {it['cite']}")
        for s in (it.get("sources") or [])[:4]:
            print(f"          raw: {s['file']} {s['loc']}")
    print(f"  reference: {q['reference_answer']}")
    print(f"  judge: {'correct' if verdict['ok'] else 'WRONG'} ({verdict['reason'][:160]}) | facts: {fact} | "
          f"citation: {cites}\n")
    query_log.append({"id": qid, "question": q["question"], "type": q["type"], "role": role, "chain": nav["chain"],
                      "nav_calls": nav["nav_calls"], "nav_seconds": nav["seconds"], "nav_usd": nav["cost_usd"],
                      "items": [i["cite"] for i in nav["items"]], "answer": a["answer"], "cited": a["cited"],
                      "answer_usd": a["cost_usd"], "judge": verdict, "fact": fact, "citation": cites})
(LOGS / "part3_queries.json").write_text(json.dumps(query_log, indent=1, ensure_ascii=False))
print("spend: navigation", spend(nav_llm), "| answers", spend(ans_llm), "| judge", spend(judge_llm))

=== Q-510 (multi_hop; multi-hop: a direct-order limit that an amendment changes later), asked as PUR-EXEC
Q: Can I raise a direct order for £40,000 of brake pads next month?
  chain: 3 model calls, 12.0 s
    read index.md
    turn 1: open [[pol-pur-001]]  (from the index)
    turn 1: open [[amd-pur-2026-03]]  (a link on [[pol-pur-001]])
    turn 2: open [[pol-apr-002]]  (a link on [[pol-pur-001]])
    turn 2: open [[role-pur-exec]]  (a link on [[pol-apr-002]])
    turn 3: done (Today is 26 October 2026, so next month is November 2026. Under POL-PUR-001 §3 and AMD-PUR-2026-03, direct purchase orders without a purchase request are permitted up to £50,000 gross (including VAT) )
  answer:
    Yes, you can raise a direct order for £40,000 next month (November 2026) [1][2]. 
    
    Key details to keep in mind:
    - **Direct order threshold:** Until 30 November 2026, direct purchase orders without a prior purchase request are permitted up to £50,000 gross value including VAT [1][2]. 
   

In [13]:
# A3 · Part 3 · 4. Lint, in three passes, on the wiki as the ingest built it (before the fixes in cell 5):
#   1. the course's code checks and the conflicts the ingest recorded (no model calls);
#   2. one check of my own: an HR-only page built from a staff document;
#   3. the course's LLM review on every page: one call per page with the course's prompt (wiki_lint.REVIEW), the page
#      body and up to 120 of its facts. The course's code reviews only pages whose sources disagree in date or
#      status; here every page is reviewed.
# Then: how many of the dump's planted problems inside my 50 files the lint found (eval/messiness.json is read only
# after the lint has run, and played no part in the lint).
import builtins
import contextlib
import re
from concurrent.futures import ThreadPoolExecutor

from ledgerline.course.l3 import wiki_lint as WLINT

t0 = time.time()
lint_rep = WLINT.lint(WIKI, DOCS, llm=None, review=False)            # pass 1
lint_man = json.loads((WIKI / "manifest.json").read_text())
for slug, p in lint_man["pages"].items():                           # pass 2
    staff_src = [f for f in p["sources"] if not f.startswith(WS.HR_DIR)]
    if p["audience"] == "hr" and staff_src:
        lint_rep["findings"].append(WLINT._f(
            "hr_page_from_staff_source", staff_src,
            f"[[{slug}]] '{p['title']}' is an HR-only page but is compiled from staff document(s) {staff_src}: "
            "staff cannot open it", "my-check", [slug]))
review_llm = CappedLLM("A3_wiki_lint", cap_usd=1.00)               # pass 3
facts_by_page = collections.defaultdict(list)
for line in (WIKI / "facts.jsonl").read_text().splitlines():
    if line.strip():
        f = json.loads(line)
        facts_by_page[f["slug"]].append(f)
resolve = WLINT.Resolver([json.loads(l) for l in (WIKI / "sources.jsonl").read_text().splitlines() if l.strip()])


def review_one(slug):  # wiki_lint.review_pages' one(), without its candidate filter
    p = lint_man["pages"][slug]
    _, body = WB.parse_page_file((WIKI / p["path"]).read_text())
    fs = facts_by_page.get(slug, [])[:120]
    ft = "\n".join(f"- ({f['file']}; {f.get('date')}; {f['status']}) {f['fact']}" for f in fs)
    r = review_llm.chat([{"role": "user", "content": WLINT.REVIEW.format(slug=slug, page=body[:14000], facts=ft[:24000])}],
                        response_format={"type": "json_object"}, **WLINT.REASONING_LOW)
    try:
        d = json.loads(re.sub(r"^```(?:json)?|```$", "", (r.content or "").strip()).strip())
        fl = d.get("findings", []) if isinstance(d, dict) else []
    except Exception:
        fl = None   # an unreadable reply is counted separately, not as "clean"
    return slug, fl


with ThreadPoolExecutor(8) as ex:
    reviewed = list(ex.map(review_one, sorted(lint_man["pages"])))
unparsed = [s for s, fl in reviewed if fl is None]
for slug, fl in reviewed:
    for x in fl or []:
        k = "red_link" if x.get("kind") == "missing_link" else str(x.get("kind", "")).strip()
        files = [g for fx in (x.get("files") or []) for g in resolve(str(fx))] + resolve(x.get("detail", ""))
        if k in WLINT.KINDS:
            lint_rep["findings"].append(WLINT._f(k, files, f"[[{slug}]]: {x.get('detail', '')}"[:400], "review", [slug]))
rev = [f for f in lint_rep["findings"] if f["source"] == "review"]
with (WIKI / "log.md").open("a") as fh:
    fh.write(f"## [{WB.now()}] lint-review | all {len(reviewed)} pages | {len(rev)} findings\n")
print(f"LLM review: {len(reviewed)} pages | unreadable replies: {unparsed or 'none'} | {spend(review_llm)}")
print(f"pages with at least one review finding: {sum(1 for _, fl in reviewed if fl)} of {len(reviewed)}")
by_kind = collections.Counter(f["kind"] for f in lint_rep["findings"])
print(f"\n{len(lint_rep['findings'])} findings in {time.time() - t0:.0f} s:", dict(by_kind.most_common()))
print("by pass:", dict(collections.Counter(f["source"] for f in lint_rep["findings"]).most_common()))
for kind in ("hr_page_from_staff_source", "contradiction", "superseded", "future_dated", "duplicate", "near_duplicate",
             "orphan", "red_link", "uncited", "citation_not_found", "inconsistent_naming"):
    fs = sorted([f for f in lint_rep["findings"] if f["kind"] == kind], key=lambda f: f["source"] != "review")
    if fs:
        print(f"\n{kind} ({len(fs)}):")
        for f in fs[:5]:
            print(f"  [{f['source']}] {f['detail'][:230]}")


@contextlib.contextmanager
def utf8_open():  # score_against_messiness opens a UTF-8 JSON file with open() and no encoding
    real = builtins.open

    def opener(file, mode="r", *args, **kwargs):
        if "b" not in mode and not args and "encoding" not in kwargs:
            kwargs["encoding"] = "utf-8"
        return real(file, mode, *args, **kwargs)

    builtins.open = opener
    try:
        yield
    finally:
        builtins.open = real


with utf8_open():
    mess = WLINT.score_against_messiness(lint_rep, DOCS)
mine = [r for r in mess["items"] if all(f in SCOPE_SRCS for f in r["files"])]
print(f"\nplanted problems whose files are all among my 50: {len(mine)}; found {sum(r['found'] for r in mine)} "
      f"(same kind or family), {sum(r['found'] or r['found_other_label'] for r in mine)} under any label")
for r in mine:
    print(f"  {r['id']} {r['kind']:15} {'found' if r['found'] else 'MISSED':6} by {r['by']}  {r['what'][:80]}")
(LOGS / "part3_lint.json").write_text(json.dumps({
    "by_kind": dict(by_kind), "review_pages": len(reviewed), "unparsed": unparsed, "spend": spend(review_llm),
    "findings": lint_rep["findings"], "planted_in_scope": mine}, indent=1, ensure_ascii=False))

LLM review: 118 pages | unreadable replies: none | {'calls': 118, 'live_calls': 2, 'replayed_calls': 116, 'live_usd': 0.0034}
pages with at least one review finding: 13 of 118

193 findings in 8 s: {'superseded': 61, 'tables_in_pdfs': 44, 'uncited': 23, 'future_dated': 18, 'duplicate': 16, 'near_duplicate': 7, 'citation_not_found': 5, 'inconsistent_naming': 5, 'orphan': 4, 'confidential': 4, 'contradiction': 3, 'legacy_abbreviations': 1, 'rule_not_in_policy': 1, 'hr_page_from_staff_source': 1}
by pass: {'code': 100, 'ingest-write': 55, 'review': 20, 'ingest-read': 17, 'my-check': 1}

hr_page_from_staff_source (1):
  [my-check] [[pol-its-008]] 'POL-ITS-008 README ACCESS' is an HR-only page but is compiled from staff document(s) ['01_policies/POL-ITS-008 IT Security Policy v3.2.pdf']: staff cannot open it

contradiction (3):
  [review] [[pol-apr-002]]: POL-APR-002 contradicts itself by stating in §3 that a document falling within the raiser's limit is automatically approved with the rais

83597

In [14]:
# A3 · Part 3 · 5. Fix three of the problems the lint found, and the bug behind the first.
# (a) POL-ITS-008, the IT Security Policy, is hidden as an HR page. Cause, in the course's wiki_sources.build_registry:
#     HR/CONFIDENTIAL/README_ACCESS.txt mentions POL-ITS-008 in its first lines, so the parser gives that file the
#     document code POL-ITS-008; the registry keeps the first file it meets for each code, and the README sorts before
#     the real policy. The page therefore got the README's HR audience and its title, the README's facts landed on it,
#     and staff lost the page. (Rajat's wiki_v4 has the same page, titled "POL-ITS-008 README ACCESS".)
#     Code fix: a code's audience and title come from the file whose own name carries the code.
#     Page fix: rewrite pol-its-008 as a staff page from the policy's own facts, and put the README's 3 facts on a new
#     HR page, hr-readme-access. Two model calls.
# (b) topic-price-tolerance has its slug as its title. Fix by hand: no model call.
# (c) sup-vpc-rot-0107 (Voltway Paints & Coatings) is an orphan. Fix by hand: link it from Voltway Cells' page, the
#     supplier it must not be confused with.
# (d) The rewritten pol-its-008 repeated its infobox as plain lines under the lead: removed in code below.
import re

if not getattr(WS.build_registry, "doc_owner_fix", False):
    _build_registry_mine = WS.build_registry

    def build_registry(srcs):
        reg = _build_registry_mine(srcs)
        for s in srcs:
            code, name = s.doc_code, s.file.rsplit("/", 1)[-1]
            slug = WS.doc_slug(code) if code else None
            if slug in reg and s.audience != "hr" and code in name and reg[slug].get("audience") == "hr":
                reg[slug].update(audience="staff", title=re.sub(r"\s+v?\d+(\.\d+)*.*$", "", s.title))
        return reg

    build_registry.my_types = build_registry.doc_owner_fix = True
    WS.build_registry = build_registry
reg_check = WS.build_registry(list(SCOPE_SRCS.values()))
print("code fix: the registry now gives pol-its-008", {k: reg_check["pol-its-008"][k] for k in ("audience", "title")})

man = json.loads((WIKI / "manifest.json").read_text())
pages = WB._pages_from_disk(WIKI, man)
reg = {s: {"type": p["type"], "title": p["title"], "audience": p["audience"]} for s, p in man["pages"].items()}
its = pages["pol-its-008"]
readme = [f for f in its["facts"] if f["file"].startswith(WS.HR_DIR)]
its["facts"] = [f for f in its["facts"] if not f["file"].startswith(WS.HR_DIR)]
its.update(sources=[f for f in its["sources"] if not f.startswith(WS.HR_DIR)], title="POL-ITS-008 IT Security Policy",
           _aud="staff", audiences={"staff"})
pages["hr-readme-access"] = {"type": "policy", "titles": collections.Counter(), "facts": readme,
                             "sources": sorted({f["file"] for f in readme}), "audiences": {"hr"}, "_aud": "hr",
                             "title": "Confidential HR folder: access notice (README_ACCESS.txt)", "status": "current"}
reg["pol-its-008"].update(audience="staff", title=its["title"])
reg["hr-readme-access"] = {"type": "policy", "title": pages["hr-readme-access"]["title"], "audience": "hr"}
for s in ("pol-its-008", "hr-readme-access"):
    pages[s]["_facts"] = WB.select_facts(pages[s]["facts"])
    pages[s]["status"] = WB._status_of([f["status"] for f in pages[s]["_facts"]])
fix_llm = CappedLLM("A3_wiki_fix", cap_usd=0.05)
b = WB.Builder(str(DOCS), WIKI, fix_llm, 2, print)
old_its = WIKI / man["pages"]["pol-its-008"]["path"]
created_its = WB.parse_page_file(old_its.read_text())[0].get("created", WB.now()[:10])
for slug in ("pol-its-008", "hr-readme-access"):
    wr = b.write_page(slug, pages[slug], pages, reg)
    wr["created"] = created_its if slug == "pol-its-008" else WB.now()[:10]
    wr["updated"] = WB.now()[:10]
    sub = "restricted" if wr["audience"] == "hr" else "pages"
    text = WB.render_page(slug, wr, wr["_refs"])
    # The model sometimes repeats the infobox as plain "key: value" lines under the lead; the infobox table above
    # already holds them, so drop the repeated block.
    repeated = "\n".join(f"{k}: {v}" for k, v in wr["infobox"].items()) + "\n\n"
    if wr["infobox"] and repeated in text:
        text = text.replace(repeated, "", 1)
        print(f"[[{slug}]]: removed the infobox lines repeated under the lead")
    (WIKI / sub / f"{slug}.md").write_text(text)
    man["pages"][slug] = {k: wr[k] for k in ("title", "type", "summary", "audience", "status", "sources", "links_out",
                                             "refs_used", "conflicts", "infobox")}
    man["pages"][slug].update(path=f"{sub}/{slug}.md", stats=wr["_stats"],
                              refs={k: {kk: wr["_refs"][k].get(kk) for kk in ("file", "page", "section", "sheet", "row",
                                                                              "loc_text", "quote", "verified", "status",
                                                                              "date")} for k in wr["refs_used"]})
old_its.unlink()   # the old copy in restricted/
facts_lines = [json.loads(l) for l in (WIKI / "facts.jsonl").read_text().splitlines() if l.strip()]
for f in facts_lines:
    if f["slug"] == "pol-its-008" and f["file"].startswith(WS.HR_DIR):
        f["slug"] = "hr-readme-access"
(WIKI / "facts.jsonl").write_text("".join(json.dumps(f, ensure_ascii=False) + "\n" for f in facts_lines))

# (b) the title
pt = WIKI / man["pages"]["topic-price-tolerance"]["path"]
pt.write_text(pt.read_text().replace('title: "topic-price-tolerance"', 'title: "Price tolerance"', 1))
man["pages"]["topic-price-tolerance"]["title"] = "Price tolerance"
# (c) the orphan
vc = WIKI / man["pages"]["sup-vc-ein-0042"]["path"]
vc.write_text(vc.read_text().replace(
    "## See also\n", "## See also\n- [[sup-vpc-rot-0107|Voltway Paints & Coatings (VPC-ROT-0107)]], a different "
    "supplier with a similar name\n", 1))
man["pages"]["sup-vc-ein-0042"]["links_out"] = sorted(set(man["pages"]["sup-vc-ein-0042"]["links_out"]) |
                                                      {"sup-vpc-rot-0107"})
(WIKI / "manifest.json").write_text(json.dumps(man, indent=1, ensure_ascii=False))
WB.write_index(WIKI, man)
with (WIKI / "log.md").open("a") as fh:
    fh.write(f"## [{WB.now()}] lint-fix | pol-its-008 -> staff page from the policy's own facts; README facts -> "
             f"hr-readme-access | topic-price-tolerance title | sup-vpc-rot-0107 linked from sup-vc-ein-0042 | repeated infobox "
             f"lines removed from pol-its-008\n")

# Check: the policy is a staff page, in index.md and openable by staff; the README page is HR-only; no orphan left
# except HR pages (which staff pages must never link to).
w = WQ.Wiki(WIKI, dump=DOCS)
inbound = collections.Counter(l for p in w.pages.values() for l in p["links_out"] if l in w.pages)
print("pol-its-008:", w.pages["pol-its-008"]["audience"], "|", w.pages["pol-its-008"]["title"], "| in index.md:",
      "[[pol-its-008]]" in (WIKI / "index.md").read_text(), "| staff may open:", w.may_open("pol-its-008", "PUR-EXEC"))
print("hr-readme-access:", w.pages["hr-readme-access"]["audience"], "| staff may open:",
      w.may_open("hr-readme-access", "PUR-EXEC"))
print("orphans now:", sorted(s for s in w.pages if inbound[s] == 0))
print("pages:", len(w.pages), "| spend:", spend(fix_llm))

code fix: the registry now gives pol-its-008 {'audience': 'staff', 'title': 'POL-ITS-008 IT Security Policy'}
[[pol-its-008]]: removed the infobox lines repeated under the lead
pol-its-008: staff | POL-ITS-008 IT Security Policy | in index.md: True | staff may open: True
hr-readme-access: hr | staff may open: False
orphans now: ['hr-conf-007', 'hr-conf-009', 'hr-conf-011', 'hr-readme-access']
pages: 119 | spend: {'calls': 2, 'live_calls': 0, 'replayed_calls': 2, 'live_usd': 0}


In [9]:
# A3 · Part 3 · 6. Update: one source document changes, and only that document is re-ingested.
# The change: POL-APR-002 Approval Matrix v3.1 (in changed_source/, made for this test from v3.0's text). It raises
# the purchase executive's limit from £250,000 to £300,000 from 26 Oct 2026 and updates example §5.1: Emma's £285,600
# order is now within her own limit. The raw dump is not touched, and the update runs on a copy of the wiki, so the
# Part 4 race uses the wiki built from the real documents.
import difflib
import shutil

from ledgerline.course.l3 import rag

CHANGED = A3 / "changed_source"
REL = "01_policies/POL-APR-002 Approval Matrix v3.1.pdf"
UPD = pathlib.Path.home() / ".cache" / "a3-wiki-update"
shutil.rmtree(UPD, ignore_errors=True)
shutil.copytree(WIKI, UPD)
upd_llm = CappedLLM("A3_wiki_update", cap_usd=0.25)
upd = WB.update([REL], str(CHANGED), UPD, upd_llm, workers=8)
print(f"wiki: {upd['pages_touched']} pages changed ({len(upd['updated'])} rewritten: {upd['updated']}; "
      f"{len(upd['created'])} new: {upd['created']}) | {upd['facts_added']} facts added | time {upd['seconds']} | "
      f"${upd['usd']:.4f} | spend {spend(upd_llm)}")
before = json.loads((WIKI / "manifest.json").read_text())["pages"]
after = json.loads((UPD / "manifest.json").read_text())["pages"]
for slug in upd["updated"]:
    a, b2 = (WIKI / before[slug]["path"]).read_text(), (UPD / after[slug]["path"]).read_text()
    d = [l for l in difflib.unified_diff(a.splitlines(), b2.splitlines(), lineterm="", n=0) if l[:1] in "+-"
         and not l.startswith(("+++", "---"))]
    hits = [l for l in b2.splitlines() if "300,000" in l]
    print(f"\n[[{slug}]]: {len(d)} lines changed; lines that now mention £300,000: {len(hits)}")
    for l in hits[:3]:
        print("   ", l[:200])

# What RAG needs for the same change (the Part 2 winner): parse and chunk the new file, embed its chunks, rebuild BM25.
# No model call. Then search: both versions are in the index, both marked CURRENT.
WINNER_ID = "structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10"
wcfg = rag.Config.from_id(WINNER_ID)
INDEX_ROOT = globals().get("INDEX_ROOT") or pathlib.Path.home() / ".cache" / "a3-rag-index-v2"
t0 = time.perf_counter()
new_units = C.parse_file(CHANGED, REL, parser="structured")
new_chunks = C.chunk(new_units, wcfg.chunker, wcfg.size, wcfg.overlap)
t_parse = time.perf_counter() - t0
emb = rag.Embedder(wcfg.embed)
emb.passages(["warm-up"])
t0 = time.perf_counter()
emb.passages([c.text for c in new_chunks])
t_embed = time.perf_counter() - t0
ridx = rag.Index.load(INDEX_ROOT / wcfg.chunking)
all_chunks = list(ridx.chunks) + list(new_chunks)
t0 = time.perf_counter()
bm_all = rag.BM25([c.text for c in all_chunks])
t_bm25 = time.perf_counter() - t0
print(f"\nRAG: parse + chunk {t_parse:.2f} s ({len(new_chunks)} chunks) | embed {t_embed * 1000:.0f} ms | BM25 rebuilt over "
      f"{len(all_chunks):,} chunks in {t_bm25:.1f} s | $0 in model calls")
QU = "What is the approval limit of a purchase executive?"
print(f"BM25 top 5 for {QU!r} after adding v3.1:")
for i, s in bm_all.search(QU, 5):
    c = all_chunks[i]
    print(f"   {s:6.2f}  {c.cite}  (version {c.meta.get('version')}, status {c.meta.get('status')})")
(LOGS / "part3_update.json").write_text(json.dumps({
    "wiki": {k: upd[k] for k in ("files", "pages_touched", "updated", "created", "seconds", "usd", "facts_added", "llm")},
    "rag": {"parse_chunk_s": round(t_parse, 3), "chunks": len(new_chunks), "embed_ms": round(t_embed * 1000, 1),
            "bm25_rebuild_s": round(t_bm25, 2), "index_chunks": len(all_chunks)}}, indent=1, ensure_ascii=False))

READ: 1 sources, 1 parts
wiki: 12 pages changed (12 rewritten: ['pol-apr-002', 'role-pur-exec', 'role-pur-sr', 'role-pur-mgr', 'role-plant-head', 'role-stores', 'err-apr-0003', 'err-apr-0009', 'user-svc-agent', 'user-eclarke', 'user-dokafor', 'user-hweber']; 0 new: []) | 23 facts added | time {'read': 20.8, 'write': 22.4, 'total': 43.3} | $0.1283 | spend {'calls': 13, 'live_calls': 13, 'replayed_calls': 0, 'live_usd': 0.1283}

[[pol-apr-002]]: 77 lines changed; lines that now mention £300,000: 2
    The gross approval limits including VAT are £300,000 for [[role-pur-exec|PUR-EXEC]] (Purchase executive), £500,000 for [[role-pur-sr|PUR-SR]] (Senior buyer), £1,000,000 for [[role-pur-mgr|PUR-MGR]] (P
    - Gross approval limits | Gross limits are £300,000 for PUR-EXEC, £500,000 for PUR-SR, £1,000,000 for PUR-MGR, and £5,000,000 for PLANT-HEAD [^r30] | Under superseded POL-APR-002 v3.0 (2026-04-01), th

[[role-pur-exec]]: 31 lines changed; lines that now mention £300,000: 6
    summary: "Pu

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


RAG: parse + chunk 0.52 s (3 chunks) | embed 72 ms | BM25 rebuilt over 13,737 chunks in 2.8 s | $0 in model calls
BM25 top 5 for 'What is the approval limit of a purchase executive?' after adding v3.1:
    28.30  approval_limits.html, § Approval limits (2016)  (version , status )
    26.41  POL-APR-002 v3.1, p.2, § 5. Examples  (version 3.1, status CURRENT)
    24.97  POL-APR-002 v3.0, p.2, § 5. Examples  (version 3.0, status CURRENT)
    24.46  Users_and_approval_limits.xlsx, Users rows 99-103  (version , status )
    24.19  Users_and_approval_limits.xlsx, Users rows 24-28  (version , status )


751

In [10]:
# A3 · Part 3 · 7. After the update: which pages still state the old £250,000 limit, and how. A page may keep it only
# as the older rule, under "Versions and conflicts" or worded as the v3.0 rule. The pages the update rewrote are
# copied to logs/part3_update_pages/ so the before and after can be read side by side.
OUT_PAGES = LOGS / "part3_update_pages"
OUT_PAGES.mkdir(exist_ok=True)
for slug in sorted(set(upd["updated"]) | set(upd["created"])):
    for tag, root, man_ in (("before", WIKI, before), ("after", UPD, after)):
        if slug in man_:
            (OUT_PAGES / f"{slug}.{tag}.md").write_text((root / man_[slug]["path"]).read_text())
for slug, p in sorted(after.items()):
    text = (UPD / p["path"]).read_text()
    old = [l.strip() for l in text.splitlines() if "250,000" in l]
    if old:
        print(f"[[{slug}]] ({'rewritten' if slug in upd['updated'] else 'NOT rewritten'}): {len(old)} line(s) with £250,000")
        for l in old:
            print("    ", l[:260])
print("pages that mentioned £250,000 before the update:",
      sorted(s for s, p in before.items() if "250,000" in (WIKI / p["path"]).read_text()))

[[pol-apr-002]] (rewritten): 1 line(s) with £250,000
     - Gross approval limits | Gross limits are £300,000 for PUR-EXEC, £500,000 for PUR-SR, £1,000,000 for PUR-MGR, and £5,000,000 for PLANT-HEAD [^r30] | Under superseded POL-APR-002 v3.0 (2026-04-01), the limit for PUR-EXEC was £250,000 [^r6], and under supersede
[[role-pur-exec]] (rewritten): 1 line(s) with £250,000
     * The purchase approval limit for PUR-EXEC is £300,000 gross including VAT per document as of 2026-10-26 | Under POL-APR-002 v3.0 (2026-04-01), the approval limit was £250,000 gross including VAT per document, and in superseded v2.0 of POL-APR-002 it was £200,
[[user-dokafor]] (rewritten): 1 line(s) with £250,000
     Daniel Okafor acts in the Ledgerline role of Purchase Manager ([[role-pur-mgr|Role PUR-MGR]]) with a standard financial approval limit of up to £1,000,000 gross [^r1]. In this role, he serves as the direct manager to [[user-eclarke|Emma Clarke]], approving pur
[[user-eclarke]] (rewritten): 2 line(s) 

In [11]:
# A3 · Part 3 · 8. The same change, measured with the Part 2 winner itself (hybrid search + reranker, top 10, with the
# confidential filter), not only BM25. The winner's index is copied to a scratch folder, so the Part 2 index is not
# touched; v3.1's chunks are added to the copy's chunk list, BM25 and Chroma; then the same questions are searched on
# the index before and after the change. No model calls.
import shutil

TMP = pathlib.Path.home() / ".cache" / "a3-rag-update-test"
shutil.rmtree(TMP, ignore_errors=True)
shutil.copytree(INDEX_ROOT / wcfg.chunking, TMP)
before_idx = rag.Index.load(INDEX_ROOT / wcfg.chunking)
base_chunks = rag.load_chunks(TMP / "chunks.jsonl.gz")
n0 = len(base_chunks)
after_idx = rag.Index(TMP, base_chunks + list(new_chunks))
new_vecs = emb.passages([c.text for c in new_chunks])
t0 = time.perf_counter()
after_idx.collection(wcfg.embed).add(
    ids=[str(n0 + j) for j in range(len(new_chunks))], embeddings=new_vecs.tolist(),
    metadatas=[rag._meta_row(c, n0 + j) for j, c in enumerate(new_chunks)], documents=[c.text for c in new_chunks])
t_chroma = time.perf_counter() - t0
t0 = time.perf_counter()
_ = after_idx.bm25
t_bm25_after = time.perf_counter() - t0
print(f"added {len(new_chunks)} chunks: Chroma add {t_chroma * 1000:.0f} ms | BM25 rebuilt in {t_bm25_after:.1f} s | "
      f"index {len(after_idx.chunks):,} chunks")

ret = rag.Retriever(after_idx, wcfg)
QUESTIONS = ["What is the approval limit of a purchase executive?",
             "Emma Clarke raises an order of £285,600 gross. Who approves it?",
             "Can a purchase executive approve an order of £280,000 on their own?"]
upd_rag = []
for qtext in QUESTIONS:
    print(f"\n=== {qtext}")
    row = {"question": qtext}
    for tag, ix in (("before", before_idx), ("after", after_idx)):
        ret.idx = ix
        out = ret.search(qtext, role="PUR-EXEC")
        ranks = {"v3.0": [], "v3.1": []}
        for it in out["items"]:
            m = it["meta"]
            if m.get("doc_id") == "POL-APR-002" and m.get("version") in ("3.0", "3.1"):
                ranks["v" + m["version"]].append(it["rank"])
        row[tag] = {"POL-APR-002 v3.0 ranks": ranks["v3.0"], "POL-APR-002 v3.1 ranks": ranks["v3.1"],
                    "top10": [it["cite"] for it in out["items"]], "ms": round(out["seconds"] * 1000, 1)}
        print(f"  {tag:6} v3.0 at ranks {ranks['v3.0'] or '-'} | v3.1 at ranks {ranks['v3.1'] or '-'} | "
              f"{out['seconds'] * 1000:.0f} ms")
        if tag == "after":
            for it in out["items"][:5]:
                m = it["meta"]
                print(f"     {it['rank']:2}. {it['cite']}  (version {m.get('version') or '-'}, status {m.get('status') or '-'})")
    upd_rag.append(row)
rep_path = LOGS / "part3_update.json"
rep = json.loads(rep_path.read_text())
rep["rag"].update(chroma_add_ms=round(t_chroma * 1000, 1), winner_search=upd_rag)
rep_path.write_text(json.dumps(rep, indent=1, ensure_ascii=False))

added 3 chunks: Chroma add 2547 ms | BM25 rebuilt in 2.7 s | index 13,737 chunks


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== What is the approval limit of a purchase executive?
  before v3.0 at ranks [2, 5] | v3.1 at ranks - | 3019 ms
  after  v3.0 at ranks [3, 6] | v3.1 at ranks [2, 7] | 142 ms
      1. approval_limits.html, § Approval limits (2016)  (version -, status -)
      2. POL-APR-002 v3.1, p.2, § 5. Examples  (version 3.1, status CURRENT)
      3. POL-APR-002 v3.0, p.2, § 5. Examples  (version 3.0, status CURRENT)
      4. Users_and_approval_limits.xlsx, Users rows 99-103  (version -, status -)
      5. Users_and_approval_limits.xlsx, Users rows 4-8  (version -, status -)

=== Emma Clarke raises an order of £285,600 gross. Who approves it?
  before v3.0 at ranks [1] | v3.1 at ranks - | 139 ms
  after  v3.0 at ranks [1] | v3.1 at ranks [2] | 132 ms
      1. POL-APR-002 v3.0, p.2, § 5. Examples  (version 3.0, status CURRENT)
      2. POL-APR-002 v3.1, p.2, § 5. Examples  (version 3.1, status CURRENT)
      3. 2026-10-23_PO_2026_01179 chargers.eml, § body  (version -, status -)
      4. POL-PUR-0

5665

## A3 · Part 4 · The race

Both methods answer my 40 questions with the same harness (`evaluate.run`): the same answer prompt, answer model
(Gemini 3.8 Flash), 250-word cap, LLM judge, exact-fact check (with the HR leak check) and citation check. Both also get
the same "withheld" notice, decided once by the course's detector, when a confidential HR document would have matched.
RAG is the Part 2 winner on the real index; the wiki is navigation over my 119-page wiki (at most 4 turns), as in the
lecture's race. Every model call is recorded in `cassettes/`.

In [15]:
# A3 · Part 4 · 1. RAG: the Part 2 winner (hybrid search + reranker, top 10) on the real index, through the course's
# race harness. Search runs one question at a time on this machine (RTX 3060); answers and judging run 8 at a time.
import random

from IPython.display import Markdown, display

from ledgerline.course.l3 import corpus as C
from ledgerline.course.l3 import evaluate as E
from ledgerline.course.l3 import rag
from ledgerline.course.l3 import wiki_query as WQ

G = G if "G" in globals() else E.Golden(DOCS)
if "SUBSET" not in globals():  # the same seeded 40 as Part 2 · 1
    SEED, PER_TYPE, ALWAYS = 3, 4, ("Q-EMMA", "Q-0412")
    by_type = collections.defaultdict(list)
    for q in G.qs:
        by_type[q["type"]].append(q)
    SUBSET = []
    for t in sorted(by_type):
        fixed = [q for q in by_type[t] if q["id"] in ALWAYS]
        rest = [q for q in by_type[t] if q["id"] not in ALWAYS]
        SUBSET += fixed + random.Random(f"{SEED}-{t}").sample(rest, PER_TYPE - len(fixed))
assert len(SUBSET) == 40
WINNER_ID = "structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10"
wcfg = rag.Config.from_id(WINNER_ID)
INDEX_ROOT = globals().get("INDEX_ROOT") or pathlib.Path.home() / ".cache" / "a3-rag-index-v2"
rag_ret = rag.Retriever(rag.Index.load(INDEX_ROOT / wcfg.chunking), wcfg)   # the real index, not Part 3's test copy
assert len(rag_ret.idx.chunks) == 13734
rag_ret.search("warm-up: load the models onto the GPU", role="PUR-EXEC")


def withheld(q, role):  # the race's notice, the same for both methods
    return rag_ret.withheld(q["question"], role)


rag_ans = CappedLLM("A3_answers", cap_usd=2.0, max_tokens=2048)
rag_judge = CappedLLM("A3_judge", cap_usd=1.0, max_tokens=512)
t0 = time.time()
race_rag = E.run(lambda question, role: rag_ret.search(question, role=role), "rag", G, qs=SUBSET, answer_llm=rag_ans,
                 judge_llm=rag_judge, workers=8, answer_words=E.RACE_WORDS, withheld=withheld)
(LOGS / "part4_rag.json").write_text(json.dumps(race_rag, indent=1, ensure_ascii=False, default=str))
o = race_rag["summary"]["overall"]
print(f"RAG: {time.time() - t0:.0f} s | answer correct {o['answer_correct']:.3f} | facts {o['fact_correct']} | "
      f"citations {o['citation_correct']} | search p95 {o['retrieval_s']['p95'] * 1000:.0f} ms | "
      f"answer p50/p95 {o['total_s']['p50']:.1f}/{o['total_s']['p95']:.1f} s | ${o['cost_per_q_usd']:.4f}/question")
print("withheld notice on:", [r["id"] for r in race_rag["records"] if r["withheld"]])
print("spend: answers", spend(rag_ans), "| judge", spend(rag_judge))

RAG: 64 s | answer correct 0.875 | facts 0.85 | citations 1.0 | search p95 149 ms | answer p50/p95 6.2/13.8 s | $0.0062/question
withheld notice on: ['Q-904', 'Q-903', 'Q-002']
spend: answers {'calls': 40, 'live_calls': 40, 'replayed_calls': 0, 'live_usd': 0.2488} | judge {'calls': 40, 'live_calls': 40, 'replayed_calls': 0, 'live_usd': 0.0383}


In [16]:
# A3 · Part 4 · 2. The wiki: navigation over my wiki, at most 4 turns, then the same answer step and judge.
# A replayed navigation call takes no time, so its recorded seconds are added back to the search time (as the lecture's
# race does); live calls are timed as they run. The raw-source index is loaded before the clock starts.
w4 = WQ.Wiki(WIKI, dump=DOCS)
_ = w4.units()
nav4 = CappedLLM("A3_wiki_nav", cap_usd=2.0, max_tokens=4000)
navigator = WQ.Navigator(w4, nav4)


def wiki_method(question, role):
    n0 = len(nav4.log)
    out = navigator.navigate(question, role)
    out["seconds"] += sum(r.seconds for r in nav4.log[n0:] if r.replayed)
    return out


wiki_ans = CappedLLM("A3_answers", cap_usd=2.0, max_tokens=2048)
wiki_judge = CappedLLM("A3_judge", cap_usd=1.0, max_tokens=512)
t0 = time.time()
race_wiki = E.run(wiki_method, "wiki", G, qs=SUBSET, answer_llm=wiki_ans, judge_llm=wiki_judge, workers=8,
                  answer_words=E.RACE_WORDS, withheld=withheld)
(LOGS / "part4_wiki.json").write_text(json.dumps(race_wiki, indent=1, ensure_ascii=False, default=str))
o = race_wiki["summary"]["overall"]
print(f"wiki: {time.time() - t0:.0f} s | answer correct {o['answer_correct']:.3f} | facts {o['fact_correct']} | "
      f"citations {o['citation_correct']} | search p95 {o['retrieval_s']['p95']:.1f} s | "
      f"answer p50/p95 {o['total_s']['p50']:.1f}/{o['total_s']['p95']:.1f} s | ${o['cost_per_q_usd']:.4f}/question")
print("spend: navigation", spend(nav4), "| answers", spend(wiki_ans), "| judge", spend(wiki_judge))

wiki: 386 s | answer correct 0.575 | facts 0.575 | citations 0.4545 | search p95 13.0 s | answer p50/p95 13.2/21.5 s | $0.0210/question
spend: navigation {'calls': 117, 'live_calls': 117, 'replayed_calls': 0, 'live_usd': 0.6146} | answers {'calls': 40, 'live_calls': 38, 'replayed_calls': 2, 'live_usd': 0.2166} | judge {'calls': 40, 'live_calls': 36, 'replayed_calls': 4, 'live_usd': 0.0316}


In [17]:
# A3 · Part 4 · 3. The results side by side, against the targets in AUDIT.md; by question type; and two cascades
# computed from the two runs (no new calls): each question's answer, time and cost come from the runs above.
def md(text):
    display(Markdown(text))


RUNS = {"RAG": race_rag, "Wiki (navigation)": race_wiki}
IN_SCOPE = {q["id"] for q in SUBSET if G.expected(q) == "answer" and G.gold(q)
            and all(g["file"].startswith(("01_policies/", "02_procedures/")) for g in G.gold(q))}


def leaks(run):
    return sum(len(r["fact"].get("leaks") or []) for r in run["records"])


rows = ["| Metric | Target | " + " | ".join(RUNS) + " |", "|---|---|" + "---:|" * len(RUNS)]
def line(name, target, f):
    rows.append(f"| {name} | {target} | " + " | ".join(f(r) for r in RUNS.values()) + " |")
S = {k: v["summary"] for k, v in RUNS.items()}
line("Answer correct (LLM judge)", "≥ 90%", lambda r: f"{r['summary']['overall']['answer_correct']:.1%}")
line("Exact-fact check", "report", lambda r: f"{r['summary']['overall']['fact_correct']:.1%} (n={r['summary']['overall']['fact_n']})")
line("Citation correct", "≥ 95%", lambda r: f"{r['summary']['overall']['citation_correct']:.1%}")
line("Confidential leaks", "0", lambda r: str(leaks(r)))
line("Retrieval recall@5", "≥ 90%", lambda r: f"{r['summary']['retrieval']['overall']['recall@5']:.1%}")
line("Search p50 / p95", "p95 ≤ 200 ms",
     lambda r: f"{r['summary']['overall']['retrieval_s']['p50'] * 1000:,.0f} / {r['summary']['overall']['retrieval_s']['p95'] * 1000:,.0f} ms")
line("Time to answer p50 / p95", "p95 ≤ 4 s",
     lambda r: f"{r['summary']['overall']['total_s']['p50']:.1f} / {r['summary']['overall']['total_s']['p95']:.1f} s")
line("Cost per question", "report", lambda r: f"${r['summary']['overall']['cost_per_q_usd']:.4f}")
line("Judge cost (all 40)", "report", lambda r: f"${r['summary']['judge_cost_usd']:.3f}")
md("\n".join(rows))

rows = ["| Type | n | " + " | ".join(f"{k} answer" for k in RUNS) + " | " + " | ".join(f"{k} time p50" for k in RUNS) + " |",
        "|---|---:|" + "---:|" * (2 * len(RUNS))]
for t in E.TYPES:
    a = [RUNS[k]["summary"]["by_type"][t] for k in RUNS]
    if a[0].get("n"):
        rows.append(f"| {t} | {a[0]['n']} | " + " | ".join(f"{x['answer_correct']:.0%}" for x in a) + " | "
                    + " | ".join(f"{x['total_s']['p50']:.1f} s" for x in a) + " |")
md("\n".join(rows))

# Scope: the wiki covers 01_policies and 02_procedures only
def acc(run, ids):
    rs = [r for r in run["records"] if r["id"] in ids]
    return f"{sum(bool(r['judge']['ok']) for r in rs)}/{len(rs)}"
answerable = {q["id"] for q in SUBSET if G.expected(q) == "answer"}
md("| Answerable questions | n | RAG | Wiki |\n|---|---:|---:|---:|\n"
   f"| gold sources all inside the wiki's 50 files | {len(IN_SCOPE)} | {acc(race_rag, IN_SCOPE)} | {acc(race_wiki, IN_SCOPE)} |\n"
   f"| gold sources (partly) outside them | {len(answerable - IN_SCOPE)} | {acc(race_rag, answerable - IN_SCOPE)} | "
   f"{acc(race_wiki, answerable - IN_SCOPE)} |")

# Cascades, computed from the two runs: the first method answers; if its answer says "not in the documents", the
# second method is asked too and its answer is used. Time and cost add up when it escalates.
def cascade(first, second):
    A = {r["id"]: r for r in first["records"]}
    B = {r["id"]: r for r in second["records"]}
    ok, ts, cost, esc = 0, [], 0.0, []
    for qid, a in A.items():
        r = a
        t, c = a["total_s"], a["cost_total_usd"]
        if E.ABSTAIN.search(a["answer"] or ""):
            esc.append(qid)
            r = B[qid]
            t += r["total_s"]
            c += r["cost_total_usd"]
        ok += bool(r["judge"]["ok"])
        ts.append(t)
        cost += c
    return {"answer_correct": ok / len(A), "p50": E.pct(ts, 50), "p95": E.pct(ts, 95), "cost": cost / len(A),
            "escalated": esc}


cas = {"RAG, then wiki": cascade(race_rag, race_wiki), "Wiki, then RAG": cascade(race_wiki, race_rag)}
md("| Cascade (computed) | Answer correct | Time p50 / p95 | Cost per question | Escalated |\n|---|---:|---:|---:|---:|\n"
   + "\n".join(f"| {k} | {v['answer_correct']:.1%} | {v['p50']:.1f} / {v['p95']:.1f} s | ${v['cost']:.4f} | "
               f"{len(v['escalated'])} |" for k, v in cas.items()))

# Per question, for RACE.md
rows = ["| Id | Type | Expected | RAG | Wiki | Wiki chain (pages opened) |", "|---|---|---|---|---|---|"]
W = {r["id"]: r for r in race_wiki["records"]}
for r in race_rag["records"]:
    q = G.by_id[r["id"]]
    wr = W[r["id"]]
    pages = ", ".join(i["cite"].split("[[")[-1].split("]]")[0] for i in wr["items"] if "wiki page" in (i["cite"] or ""))
    rows.append(f"| {r['id']} | {r['type']} | {G.expected(q)} | {'✓' if r['judge']['ok'] else '✗'} | "
                f"{'✓' if wr['judge']['ok'] else '✗'} | {pages or '-'} |")
md("\n".join(rows))
(LOGS / "part4_summary.json").write_text(json.dumps({
    "summary": {k: v["summary"] for k, v in RUNS.items()}, "leaks": {k: leaks(v) for k, v in RUNS.items()},
    "in_scope_ids": sorted(IN_SCOPE), "cascades": cas}, indent=1, ensure_ascii=False, default=str))

| Metric | Target | RAG | Wiki (navigation) |
|---|---|---:|---:|
| Answer correct (LLM judge) | ≥ 90% | 87.5% | 57.5% |
| Exact-fact check | report | 85.0% (n=40) | 57.5% (n=40) |
| Citation correct | ≥ 95% | 100.0% | 45.5% |
| Confidential leaks | 0 | 0 | 0 |
| Retrieval recall@5 | ≥ 90% | 90.9% | 51.5% |
| Search p50 / p95 | p95 ≤ 200 ms | 139 / 149 ms | 7,906 / 12,993 ms |
| Time to answer p50 / p95 | p95 ≤ 4 s | 6.2 / 13.8 s | 13.2 / 21.5 s |
| Cost per question | report | $0.0062 | $0.0210 |
| Judge cost (all 40) | report | $0.038 | $0.036 |

| Type | n | RAG answer | Wiki (navigation) answer | RAG time p50 | Wiki (navigation) time p50 |
|---|---:|---:|---:|---:|---:|
| single_lookup | 4 | 100% | 75% | 4.0 s | 9.6 s |
| procedure | 4 | 100% | 50% | 7.7 s | 16.7 s |
| rule | 4 | 100% | 100% | 4.8 s | 9.5 s |
| service_field | 4 | 100% | 50% | 4.5 s | 13.6 s |
| table_lookup | 4 | 100% | 25% | 6.2 s | 13.9 s |
| multi_hop | 4 | 25% | 25% | 9.9 s | 17.2 s |
| version_conflict | 4 | 75% | 75% | 7.4 s | 13.3 s |
| exact_code | 4 | 100% | 0% | 7.9 s | 12.6 s |
| not_in_docs | 4 | 100% | 100% | 4.4 s | 13.2 s |
| permission | 4 | 75% | 75% | 7.2 s | 15.4 s |

| Answerable questions | n | RAG | Wiki |
|---|---:|---:|---:|
| gold sources all inside the wiki's 50 files | 13 | 12/13 | 11/13 |
| gold sources (partly) outside them | 20 | 17/20 | 6/20 |

| Cascade (computed) | Answer correct | Time p50 / p95 | Cost per question | Escalated |
|---|---:|---:|---:|---:|
| RAG, then wiki | 90.0% | 6.8 / 20.6 s | $0.0096 | 6 |
| Wiki, then RAG | 85.0% | 16.7 / 27.3 s | $0.0234 | 17 |

| Id | Type | Expected | RAG | Wiki | Wiki chain (pages opened) |
|---|---|---|---|---|---|
| Q-0412 | exact_code | answer | ✓ | ✗ | err-po-0412 |
| Q-705 | exact_code | answer | ✓ | ✗ | err-sup-0118 |
| Q-703 | exact_code | answer | ✓ | ✗ | err-po-0610 |
| Q-711 | exact_code | answer | ✓ | ✗ | pol-its-008, sop-it-001, sop-it-002 |
| Q-EMMA | multi_hop | answer | ✗ | ✗ | pol-sup-003, pol-pur-001, sop-pur-001, items-bat, sop-pur-006, sup-vc-ein-0042, sop-pur-004 |
| Q-502 | multi_hop | answer | ✓ | ✗ | items-bat, pol-pur-001, pol-sup-003, sop-pur-006, sup-vc-ein-0042, sop-pur-001, sop-pur-004, amd-sup-2026-02 |
| Q-510 | multi_hop | answer | ✗ | ✓ | amd-pur-2026-03, pol-pur-001, sop-pur-002, pol-apr-002, role-pur-exec |
| Q-505 | multi_hop | answer | ✗ | ✗ | sop-pur-005, pol-sup-003, amd-sup-2026-02 |
| Q-811 | not_in_docs | abstain | ✓ | ✓ | sup-vc-ein-0042, items-bat, sop-pur-004, sup-vc-ein-0009, sop-pur-006, pol-pur-001, pol-bat-004, pol-sup-003 |
| Q-809 | not_in_docs | abstain | ✓ | ✓ | pol-its-008, sop-it-001, sop-it-002 |
| Q-808 | not_in_docs | abstain | ✓ | ✓ | pol-drp-009, pol-its-008, pol-hr-020 |
| Q-802 | not_in_docs | abstain | ✓ | ✓ | pol-its-008, pol-hr-020, data-warehouses, sop-it-001 |
| Q-907 | permission | answer | ✓ | ✓ | hr-conf-007 |
| Q-904 | permission | refuse | ✓ | ✓ | pol-hr-020, pol-bat-004, sop-qua-002, sop-inv-004, pol-hse-010, user-slindqvist, user-cshaw, user-pjansen |
| Q-903 | permission | refuse | ✓ | ✓ | sop-fin-001, data-warehouses, role-stores, pol-hr-020 |
| Q-906 | permission | refuse | ✗ | ✗ | pol-hr-020, pol-pur-001, pol-abc-011, role-pur-exec, role-pur-mgr, role-pur-sr |
| Q-110 | procedure | answer | ✓ | ✗ | - |
| Q-104 | procedure | answer | ✓ | ✗ | sop-pur-002, screen-pur-po-01, sop-pur-003 |
| Q-112 | procedure | answer | ✓ | ✓ | sop-pur-002, pol-pur-001, screen-pur-po-01, screen-pur-por-05 |
| Q-118 | procedure | answer | ✓ | ✓ | sop-inv-004, pol-bat-004, sop-prd-001, screen-inv-mrn-08, items-bat |
| Q-219 | rule | answer | ✓ | ✓ | pol-ret-006, sop-sal-002 |
| Q-222 | rule | answer | ✓ | ✓ | pol-drp-009, pol-sup-003, sop-pur-005 |
| Q-208 | rule | answer | ✓ | ✓ | amd-sup-2026-02, pol-sup-003, sop-pur-005 |
| Q-213 | rule | answer | ✓ | ✓ | pol-apr-002, pol-pur-001, role-plant-head |
| Q-312 | service_field | answer | ✓ | ✓ | sop-pur-001, pol-sup-003, err-sup-0118 |
| Q-322 | service_field | answer | ✓ | ✓ | sop-pur-002, screen-pur-po-01 |
| Q-334 | service_field | answer | ✓ | ✗ | sop-ap-001, sop-it-002, screen-ap-inv-01, screen-pur-po-01, screen-ap-bnk-06, screen-sys-api-14, sop-pur-002 |
| Q-335 | service_field | answer | ✓ | ✗ | sop-pur-003, sop-it-002, pol-drp-009, sop-pur-002, screen-pur-pom-06, screen-pur-po-01 |
| Q-002 | single_lookup | answer | ✓ | ✓ | pol-its-008, sop-it-001 |
| Q-011 | single_lookup | answer | ✓ | ✓ | pol-bat-004, sop-inv-004, pol-hse-010 |
| Q-010 | single_lookup | answer | ✓ | ✗ | pol-its-008, sop-it-002, pol-drp-009, user-svc-agent, sop-ap-002 |
| Q-020 | single_lookup | answer | ✓ | ✓ | sop-inv-003 |
| Q-412 | table_lookup | answer | ✓ | ✗ | sop-pur-004, topic-price-tolerance, pol-sup-003, sop-pur-002, sop-pur-006, pol-pur-001, items-bat, screen-pur-po-01 |
| Q-422 | table_lookup | answer | ✓ | ✓ | data-warehouses, sop-inv-004, pol-bat-004 |
| Q-413 | table_lookup | answer | ✓ | ✗ | sop-pur-004, topic-price-tolerance, sop-pur-002, sop-pur-006, pol-pur-001, screen-pur-po-01 |
| Q-433 | table_lookup | answer | ✓ | ✗ | items-bat, data-warehouses, items-pkg |
| Q-618 | version_conflict | answer | ✗ | ✓ | pol-tex-007 |
| Q-610 | version_conflict | answer | ✓ | ✓ | pol-tex-007 |
| Q-615 | version_conflict | answer | ✓ | ✗ | amd-sup-2026-02, pol-sup-003, sop-pur-005 |
| Q-607 | version_conflict | answer | ✓ | ✓ | sop-pur-002, screen-pur-po-01, err-scr-0221 |

21441

## A3 · Part 5 · My own domain

The domain is my A1 capstone's: Ask Northbridge for accounts payable at Northbridge Cycles Pvt. Ltd. (Pune). The
documents and questions are invented (`domain/README.md`), modelled on real accounts payable documents: 41 files
(PDF, DOCX, XLSX, e-mail, Markdown), 2 of them confidential, with planted mess (an old policy version beside the
current one, an e-mail amending a limit from 1 December, a duplicated procedure, two invoices with the same number).

The 12 documentation questions are asked the way their users would type them (`part5/golden_v2.jsonl`). The cells
follow Lecture 3's twelve rules:
1. Cell 1: the audit, versions and effective dates, and the golden set (rules 1, 4 and 10).
2. Cell 2: the retrieval grid, re-run on this domain (rules 5 to 7 and 12).
3. Cell 3: exact codes (rule 8).
4. Cell 4: the 12 questions through Part 4's harness, for the Part 4 winner (and this domain's pick, if it
   differs), against targets set for accounts payable (rules 3 and 9).
5. Cell 5: the check script for CI (rule 12).

DOMAIN.md has the results.

In [5]:
# A3 · Part 5 · 1. My own domain: accounts payable at Northbridge Cycles Pvt. Ltd. (Pune), from my A1 capstone.
# Rules 1, 4 and 10: check and audit the documents, make versions and effective dates explicit, and load the 12
# documentation questions, in the words their users would type. Nothing here calls a model.
import importlib
import re

from ledgerline.course.l3 import corpus as C
from ledgerline.course.l3 import evaluate as E
from ledgerline.course.l3 import rag

assert "CappedLLM" in globals(), "run 'A3 · Part 3 · 0' first (cassettes, key, spending cap)"
DOM, LOGS, PART5 = A3 / "domain", A3 / "logs", A3 / "part5"
DOM_DOCS = DOM / "docs"
if str(PART5) not in sys.path:
    sys.path.insert(0, str(PART5))
import domain_rules as DR

importlib.reload(DR)
WINNER_ID = "structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10"   # the Part 4 winner

# Checks before anything is measured: every file parses, and the two HR files are flagged confidential.
dom_files = DR.files(DOM_DOCS)
t0 = time.time()
units_raw = DR.parse(DOM_DOCS)
assert len(dom_files) == 41 and {u.file for u in units_raw} == set(dom_files), "a file is missing or produced no text"
dom_conf = sorted({u.file for u in units_raw if u.meta.get("confidential")})
assert dom_conf == [f for f in dom_files if "/HR/CONFIDENTIAL/" in f] and len(dom_conf) == 2, dom_conf
print(f"{len(dom_files)} files -> {len(units_raw)} units in {time.time() - t0:.1f} s | confidential: "
      f"{[f.split('/')[-1] for f in dom_conf]}")

# Rule 1 · Audit: files, PDF pages, spreadsheet rows and tokens, by folder and by class (o200k tokens, structured parse).
aud = collections.defaultdict(lambda: {"files": set(), "pages": set(), "rows": 0, "tokens": 0})
for u in units_raw:
    for key in (u.file.split("/")[0], C.FMT_CLASS.get(u.fmt, "unstructured")):
        a = aud[key]
        a["files"].add(u.file)
        if u.loc.get("page"):
            a["pages"].add((u.file, u.loc["page"]))
        a["rows"] += u.fmt in ("xlsx", "csv")
        a["tokens"] += C.n_tokens(u.text)
rows = ["| Folder or class | Files | PDF pages | Spreadsheet rows | Tokens |", "|---|---:|---:|---:|---:|"]
for k in sorted(aud, key=lambda k: (k in C.FMT_CLASS.values(), k)):
    a = aud[k]
    rows.append(f"| {k} | {len(a['files'])} | {len(a['pages'])} | {a['rows']} | {a['tokens']:,} |")
rows.append(f"| **all** | {len(dom_files)} | {len({(u.file, u.loc['page']) for u in units_raw if u.loc.get('page')})} | "
            f"{sum(u.fmt in ('xlsx', 'csv') for u in units_raw)} | {sum(C.n_tokens(u.text) for u in units_raw):,} |")
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))

# Rule 10 · Versions and effective dates, made explicit (see domain_rules.handle_versions).
units_v, vrep = DR.handle_versions(units_raw, DR.TODAY)
print("today:", DR.TODAY, "| effective dates read:", sum(bool(m.get("effective")) for m in vrep["metas"].values()),
      "of", len(vrep["metas"]), "files")
print("superseded (older version still in force):",
      {f.split("/")[-1]: "-> " + n.split("/")[-1] for f, n in vrep["superseded"].items()})
print("not yet in force:", [f.split("/")[-1] for f in vrep["future"]])
print("spreadsheet rows marked:", [(f.split("/")[-1], l, s) for f, l, s in vrep["rows"]])
print("shown to the answer model as:", {f.split("/")[-1]: C.status_line(vrep["metas"][f]) for f in vrep["superseded"]})

# Rule 4 · The golden set: 12 documentation questions with reference answers; every gold quote checked at its location.
DQ = DR.load_golden(PART5 / "golden_v2.jsonl")
DFACTS = DR.confidential_facts(units_raw, DOM / "facts.jsonl")   # the leak check's values: the confidential files' amounts
DG = DR.DomainGolden(DOM_DOCS, DQ, DFACTS)
E.ROLE_NAMES.update({code: name for code, name in DR.ROLES.values()})
assert not {q["asker_role"] for q in DQ} & C.HR_ROLES
for q in DQ:
    for g in q["gold_sources"]:
        here = [u for u in units_raw if u.file == g["file"] and (
            (g.get("page") and u.loc.get("page") == g["page"]) or (g.get("row") and u.loc.get("row") == g["row"]
                                                                  and u.loc.get("sheet") == g.get("sheet"))
            or (not g.get("page") and not g.get("row")))]
        assert any(" ".join(g["quote"].split()) in " ".join(u.text.split()) for u in here), (q["id"], g["file"])
lines = ["| Id | Type | Asked by | Expected | Question |", "|---|---|---|---|---|"]
for q in DQ:
    lines.append(f"| {q['id']} | {q['type']} | {q['asker']} | {DG.expected(q)} | {q['question']} |")
show_table("\n".join(lines)) if "show_table" in globals() else print("\n".join(lines))
print(f"{len(DQ)} questions; every gold quote found at its page, section or row; "
      f"{sum(DG.expected(q) == 'answer' for q in DQ)} scored for recall")

41 files -> 248 units in 1.1 s | confidential: ['HR-CONF-003 Finance Salary Bands FY2026-27.xlsx', 'HR-CONF-005 Bank Mandate and Authorised Signatories.pdf']
| Folder or class | Files | PDF pages | Spreadsheet rows | Tokens |
|---|---:|---:|---:|---:|
| 01_policies | 11 | 15 | 3 | 4,386 |
| 02_procedures | 11 | 0 | 0 | 1,492 |
| 03_ledgerline_manuals | 2 | 3 | 0 | 967 |
| 04_forms | 3 | 0 | 0 | 242 |
| 05_master_data | 8 | 0 | 48 | 1,670 |
| 06_transactions | 1 | 0 | 7 | 460 |
| 07_correspondence | 5 | 0 | 0 | 514 |
| semi-structured | 5 | 0 | 0 | 514 |
| structured | 10 | 0 | 58 | 2,242 |
| unstructured | 26 | 18 | 0 | 6,975 |
| **all** | 41 | 18 | 58 | 9,731 |
today: 2026-11-09 | effective dates read: 15 of 41 files
superseded (older version still in force): {'POL-AP-001 Accounts Payable Policy v2.0.pdf': '-> POL-AP-001 Accounts Payable Policy v2.1.pdf'}
not yet in force: []
spreadsheet rows marked: [('RCM Services Table.xlsx', {'sheet': 'Services', 'row': 8}, 'expired')]
shown to th

In [6]:
# A3 · Part 5 · 2. Rules 5 to 7 and 12: the grid from Part 2, re-run on this domain. Retrieval only (no model calls, $0).
# The rule, fixed before the grid runs: the highest recall@5 on the 10 scored questions, among configurations whose
# search p95 is at most 200 ms on this machine; an exact tie goes to the higher MRR, then the lower p95.
EMBEDS5 = ["bge-small", "bge-base", "e5-small", "gte-small"]   # rule 7: measured here, not carried over
SIZES5 = (200, 400, 800)
LIMIT_MS = 200
INDEX5 = pathlib.Path.home() / ".cache" / "a3-domain-index-p5"
scored = [q for q in DQ if DG.expected(q) == "answer"]

import hashlib


def open_index5(ch, path):
    """Reuse the index at path if it was built from exactly these chunks with these models; else build it. (On Windows a
    Chroma folder the kernel still has open cannot be deleted, so rebuilding over it fails.)"""
    marker = path / "built.json"
    sig = {"models": EMBEDS5, "chunks": hashlib.sha1("\n".join(c.cid + c.text for c in ch).encode()).hexdigest()}
    if marker.exists() and json.loads(marker.read_text(encoding="utf-8")) == sig:
        return rag.Index.load(path)
    idx = rag.Index.build(ch, EMBEDS5, path)
    marker.write_text(json.dumps(sig), encoding="utf-8")
    return idx


t0 = time.time()
idx5, rows5 = {}, []
for size in SIZES5:
    ch = C.chunk(units_v, "structure", size, 0.15)
    idx5[size] = open_index5(ch, INDEX5 / C.chunking_id("structured", "structure", size, 0.15))
    for retrieval in ("bm25", "dense", "hybrid"):
        for emb in (["bge-small"] if retrieval == "bm25" else EMBEDS5):
            for rr in (False, True):
                cfg = rag.Config("structured", "structure", size, 0.15, emb, retrieval, rr, False, 10)
                ret5 = rag.Retriever(idx5[size], cfg)
                ret5.search("warm-up", role="AP-EXEC")
                ru = E.retrieval_only(lambda question, role: ret5.search(question, role=role), DG, qs=scored)
                su = ru["summary"]
                rows5.append({"id": cfg.id, "size": size, "retrieval": retrieval, "embed": "-" if retrieval == "bm25" else emb,
                              "rerank": rr, "chunks": len(ch), "recall@5": su["overall"]["recall@5"],
                              "recall@1": su["overall"]["recall@1"], "mrr": su["overall"]["mrr"],
                              "p95_ms": round(su["latency_s"]["p95"] * 1000, 1),
                              "ranks": {r["id"]: r["retrieval"]["rank"] for r in ru["records"]}})
print(f"{len(rows5)} configurations in {time.time() - t0:.0f} s (index builds and model downloads included)")

eligible5 = [r for r in rows5 if r["p95_ms"] <= LIMIT_MS]
WIN5 = max(eligible5, key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
part2 = next(r for r in rows5 if r["id"] == WINNER_ID)
print(f"{len(eligible5)} of {len(rows5)} have search p95 <= {LIMIT_MS} ms")
print("WINNER:", WIN5["id"], f"| recall@5 {WIN5['recall@5']:.2f} | MRR {WIN5['mrr']:.3f} | p95 {WIN5['p95_ms']} ms")
print("Part 2's winner here:", part2["id"], f"| recall@5 {part2['recall@5']:.2f} | MRR {part2['mrr']:.3f} | p95 {part2['p95_ms']} ms")

rows = ["| Configuration | Chunks | Recall@1 | Recall@5 | MRR | Search p95 |", "|---|---:|---:|---:|---:|---:|"]
for r in sorted(rows5, key=lambda r: (-r["recall@5"], -r["mrr"], r["p95_ms"]))[:15] + [part2]:
    rows.append(f"| {r['id']}{' **(winner)**' if r is WIN5 else ' (Part 2 winner)' if r is part2 else ''} | {r['chunks']} | "
                f"{r['recall@1']:.2f} | {r['recall@5']:.2f} | {r['mrr']:.3f} | {r['p95_ms']} ms |")
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))


def by(key):  # each setting's median over the configurations that have it
    out = collections.defaultdict(list)
    for r in rows5:
        out[r[key]].append(r)
    return {k: (statistics.median(x["recall@5"] for x in v), statistics.median(x["mrr"] for x in v), len(v))
            for k, v in out.items()}


rows = ["| Setting | Value | Configurations | Median recall@5 | Median MRR |", "|---|---|---:|---:|---:|"]
for key in ("size", "retrieval", "embed", "rerank"):
    for k, (r5, mrr, n) in sorted(by(key).items(), key=lambda x: str(x[0])):
        rows.append(f"| {key} | {k} | {n} | {r5:.2f} | {mrr:.3f} |")
print("configurations with recall@5 = 1.00:", sum(r["recall@5"] == 1.0 for r in rows5), "of", len(rows5))
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))
_ = (LOGS / "part5_grid.json").write_text(json.dumps({"rule": "max recall@5 with search p95 <= 200 ms; ties: MRR, then p95",
                                                      "winner": WIN5["id"], "rows": rows5}, indent=1, default=str),
                                          encoding="utf-8")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

54 configurations in 59 s (index builds and model downloads included)
54 of 54 have search p95 <= 200 ms
WINNER: structured-structure-400-o15|gte-small|dense|rr1|qr0|k10 | recall@5 1.00 | MRR 0.925 | p95 130.3 ms
Part 2's winner here: structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10 | recall@5 1.00 | MRR 0.820 | p95 128.0 ms
| Configuration | Chunks | Recall@1 | Recall@5 | MRR | Search p95 |
|---|---:|---:|---:|---:|---:|
| structured-structure-400-o15|gte-small|dense|rr1|qr0|k10 **(winner)** | 51 | 0.90 | 1.00 | 0.925 | 130.3 ms |
| structured-structure-400-o15|bge-small|dense|rr1|qr0|k10 | 51 | 0.80 | 1.00 | 0.875 | 129.8 ms |
| structured-structure-400-o15|e5-small|dense|rr1|qr0|k10 | 51 | 0.80 | 1.00 | 0.870 | 130.2 ms |
| structured-structure-400-o15|bge-base|dense|rr1|qr0|k10 | 51 | 0.70 | 1.00 | 0.825 | 128.9 ms |
| structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10 (Part 2 winner) | 51 | 0.70 | 1.00 | 0.820 | 128.0 ms |
| structured-structure-400-o15|bge-base|hy

In [7]:
# A3 · Part 5 · 3. Rule 8: exact codes typed on their own, as a user pastes them from a screen. Retrieval only ($0).
# The 10 scored questions hold only a few codes, too few to show whether dense search handles them, and cell 2's rule
# can pick a dense-only configuration. Every code-like string in the staff documents (letters, a hyphen, at least one
# digit) is searched alone; a hit = a chunk among the top 5 contains that code. The decision rule is stated here, before the
# numbers: if the dense-only pick finds fewer codes than the best hybrid or BM25 configuration, rule 8 applies and the
# pick is redone among hybrid and BM25 configurations by cell 2's rule (recall@5, then MRR, then p95).
CODE_RE = re.compile(r"\b[A-Z][A-Z0-9]*(?:-[A-Z0-9]+)+\b")
staff_text = {}
for u in units_v:
    if not u.meta.get("confidential"):
        staff_text.setdefault(u.file, []).append(u.text)
CODES = sorted({c for ts in staff_text.values() for t in ts for c in CODE_RE.findall(t) if re.search(r"\d", c)})
print(f"{len(CODES)} distinct codes in the staff documents, e.g. {CODES[:8]}")


def code_recall(cfg):
    r = rag.Retriever(idx5[cfg.size], cfg)
    r.search("warm-up", role="AP-EXEC")
    hits, missed = 0, []
    for code in CODES:
        items = r.search(code, role="AP-EXEC")["items"][:5]
        if any(code in (it.get("text") or "") for it in items):
            hits += 1
        else:
            missed.append(code)
    return hits / len(CODES), missed


# Candidates: every configuration at the pick's chunk size that uses the reranker, plus BM25 alone with and without it.
size5 = rag.Config.from_id(WIN5["id"]).size
cand = {r["id"]: r for r in rows5 if r["size"] == size5 and (r["rerank"] or r["retrieval"] == "bm25")}
code_rows = {}
for cid in cand:
    rate, missed = code_recall(rag.Config.from_id(cid))
    code_rows[cid] = {"code_recall@5": rate, "missed": missed}
rows = ["| Configuration | Codes found in top 5 | Missed (first 6) | Questions recall@5 / MRR |",
        "|---|---:|---|---:|"]
for cid, v in sorted(code_rows.items(), key=lambda x: -x[1]["code_recall@5"]):
    q = cand[cid]
    rows.append(f"| {cid}{' **(cell 2 pick)**' if cid == WIN5['id'] else ''} | {v['code_recall@5']:.0%} "
                f"({round(v['code_recall@5'] * len(CODES))}/{len(CODES)}) | {', '.join(v['missed'][:6])} | "
                f"{q['recall@5']:.2f} / {q['mrr']:.3f} |")
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))

pick_codes = code_rows[WIN5["id"]]["code_recall@5"]
best_lex = max(v["code_recall@5"] for cid, v in code_rows.items() if cand[cid]["retrieval"] in ("bm25", "hybrid"))
if WIN5["retrieval"] == "dense" and pick_codes < best_lex:
    lexical = [r for r in rows5 if r["retrieval"] in ("bm25", "hybrid") and r["p95_ms"] <= LIMIT_MS]
    WIN5_DENSE = WIN5
    WIN5 = max(lexical, key=lambda r: (r["recall@5"], r["mrr"], -r["p95_ms"]))
    print(f"Rule 8 applies: the dense-only pick finds {pick_codes:.0%} of codes, the best hybrid/BM25 {best_lex:.0%}.")
    print("NEW PICK:", WIN5["id"], f"| recall@5 {WIN5['recall@5']:.2f} | MRR {WIN5['mrr']:.3f} | p95 {WIN5['p95_ms']} ms")
else:
    print(f"Rule 8 does not change the pick: it finds {pick_codes:.0%} of codes, the best hybrid/BM25 {best_lex:.0%}.")
g5 = json.loads((LOGS / "part5_grid.json").read_text(encoding="utf-8"))
g5.update({"winner": WIN5["id"], "codes": CODES, "code_recall": code_rows,
           "rule8": "if the dense-only pick finds fewer codes than the best hybrid/BM25, re-pick among hybrid/BM25"})
_ = (LOGS / "part5_grid.json").write_text(json.dumps(g5, indent=1, default=str), encoding="utf-8")

46 distinct codes in the staff documents, e.g. ['CC-2290', 'CC-CHG-0018', 'DEC-SR-2026-014', 'ERR-AP-0107', 'ERR-AP-0212', 'ERR-AP-0215', 'ERR-BNK-0602', 'ERR-GST-0418']
| Configuration | Codes found in top 5 | Missed (first 6) | Questions recall@5 / MRR |
|---|---:|---|---:|
| structured-structure-400-o15|-|bm25|rr0|qr0|k10 | 100% (46/46) |  | 0.70 / 0.494 |
| structured-structure-400-o15|-|bm25|rr1|qr0|k10 | 100% (46/46) |  | 0.80 / 0.717 |
| structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10 | 100% (46/46) |  | 1.00 / 0.820 |
| structured-structure-400-o15|bge-base|hybrid|rr1|qr0|k10 | 100% (46/46) |  | 1.00 / 0.820 |
| structured-structure-400-o15|e5-small|hybrid|rr1|qr0|k10 | 100% (46/46) |  | 0.90 / 0.817 |
| structured-structure-400-o15|gte-small|hybrid|rr1|qr0|k10 | 100% (46/46) |  | 1.00 / 0.820 |
| structured-structure-400-o15|bge-small|dense|rr1|qr0|k10 | 98% (45/46) | FY2026-27 | 1.00 / 0.875 |
| structured-structure-400-o15|bge-base|dense|rr1|qr0|k10 | 98% (45/46) |

In [10]:
# A3 · Part 5 · 4. The 12 documentation questions through the same harness as Part 4 (evaluate.run): the same answer
# model, 250-word cap, judge, exact-fact check, leak check, citation check and "withheld" notice. It runs the Part 4
# winner, unchanged (what the brief asks), and also the pick of cells 2 and 3 if that differs. Rules 3 and 9: the
# targets are the ones set for accounts payable. One change to the harness: the first two lines of the answer prompt
# name this domain's company and its "today" (9 November 2026) instead of Manchester and 26 October; the rest of the
# prompt is the course's, word for word. Every model call is recorded in cassettes/.
COURSE_SYSTEM = E.ANSWER_SYSTEM
DOMAIN_SYSTEM = COURSE_SYSTEM.replace(
    "the documentation assistant of Northbridge Cycles (Manchester), who run Ledgerline ERP 7.2.\n"
    "Today is Monday 26 October 2026.",
    "the accounts payable documentation assistant of Northbridge Cycles Pvt. Ltd. (Pune), who run Ledgerline.\n"
    "Today is Monday 9 November 2026.")
assert DOMAIN_SYSTEM != COURSE_SYSTEM, "the course's answer prompt has changed: check the replace above"
TARGETS5 = {"recall@5": 0.90, "answer": 0.90, "citations": 0.95, "search_p95_ms": 200, "answer_p95_s": 10.0}
MUST_BE_RIGHT = {"D-04": "payment status", "D-10": "permission"}   # the capstone's two never-events
CONFIGS5 = {"Part 4 winner": WINNER_ID}
if WIN5["id"] != WINNER_ID:
    CONFIGS5["Part 5 pick"] = WIN5["id"]
else:
    print("The pick of cells 2 and 3 is the Part 4 winner itself, so it runs once.")

E.ANSWER_SYSTEM = DOMAIN_SYSTEM
try:
    runs5 = {}
    for name, cid in CONFIGS5.items():
        cfg = rag.Config.from_id(cid)
        ret = rag.Retriever(idx5[cfg.size], cfg)
        ret.search("warm-up", role="AP-EXEC")
        ans = CappedLLM("A3_domain_answers", cap_usd=0.60, max_tokens=2048)
        jud = CappedLLM("A3_domain_judge", cap_usd=0.30, max_tokens=512)
        runs5[name] = E.run(lambda question, role: ret.search(question, role=role), name, DG, qs=DQ, answer_llm=ans,
                            judge_llm=jud, workers=8, answer_words=E.RACE_WORDS,
                            withheld=lambda q, role: ret.withheld(q["question"], role))
        runs5[name]["config"] = cid
        runs5[name]["spend"] = {"answers": spend(ans), "judge": spend(jud)}
finally:
    E.ANSWER_SYSTEM = COURSE_SYSTEM


def ok_ids(run):
    return {r["id"] for r in run["records"] if r["judge"]["ok"]}


rows = ["| Metric | Target (accounts payable) | " + " | ".join(CONFIGS5) + " |", "|---|---|" + "---:|" * len(CONFIGS5)]
def line(name, target, f):
    rows.append(f"| {name} | {target} | " + " | ".join(f(runs5[k]) for k in CONFIGS5) + " |")
line("Configuration", "", lambda r: r["config"])
line("Retrieval recall@5", "≥ 90%", lambda r: f"{r['summary']['retrieval']['overall']['recall@5']:.0%} (n={r['summary']['retrieval']['overall']['n']})")
line("Recall@1 / @3, MRR", "report", lambda r: f"{r['summary']['retrieval']['overall']['recall@1']:.0%} / {r['summary']['retrieval']['overall']['recall@3']:.0%}, {r['summary']['retrieval']['overall']['mrr']:.2f}")
line("Answer correct (LLM judge)", "≥ 90%", lambda r: f"{r['summary']['overall']['answer_correct']:.0%} ({len(ok_ids(r))}/12)")
line("Payment-status and permission answers", "all right", lambda r: f"{sum(i in ok_ids(r) for i in MUST_BE_RIGHT)}/{len(MUST_BE_RIGHT)}")
line("Exact-fact check", "report", lambda r: f"{r['summary']['overall']['fact_correct']:.0%} (n={r['summary']['overall']['fact_n']})")
line("Judge vs exact check", "report", lambda r: f"agree {r['summary']['judge_vs_fact']['agree']:.0%}, kappa {r['summary']['judge_vs_fact']['kappa']}")
line("Citation correct", "≥ 95%", lambda r: f"{r['summary']['overall']['citation_correct']:.0%}")
line("Confidential leaks", "0", lambda r: str(sum(len(x["fact"].get("leaks") or []) for x in r["records"])))
line("Search p50 / p95", "p95 ≤ 200 ms", lambda r: f"{r['summary']['overall']['retrieval_s']['p50'] * 1000:.0f} / {r['summary']['overall']['retrieval_s']['p95'] * 1000:.0f} ms")
line("Time to answer p50 / p95", "p95 ≤ 10 s", lambda r: f"{r['summary']['overall']['total_s']['p50']:.1f} / {r['summary']['overall']['total_s']['p95']:.1f} s")
line("Cost per question", "report", lambda r: f"${r['summary']['overall']['cost_per_q_usd']:.4f}")
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))

RS = {k: {r["id"]: r for r in runs5[k]["records"]} for k in CONFIGS5}
LAST = list(CONFIGS5)[-1]
rk = lambda r: (r["retrieval"]["rank"] or "-") if r["retrieval"]["n_gold"] else "n/a"
rows = [f"| Id | Type | Question | Gold rank ({' / '.join(CONFIGS5)}) | Answer ({' / '.join(CONFIGS5)}) | Withheld notice | "
        f"Judge's reason ({LAST}) |", "|---|---|---|---:|---|---|---|"]
for q in DQ:
    rs = [RS[k][q["id"]] for k in CONFIGS5]
    rows.append(f"| {q['id']} | {q['type']} | {q['question']} | {' / '.join(str(rk(r)) for r in rs)} | "
                f"{' / '.join('✓' if r['judge']['ok'] else '✗' for r in rs)} | {'yes' if rs[-1]['withheld'] else ''} | "
                f"{rs[-1]['judge']['reason']} |")
show_table("\n".join(rows)) if "show_table" in globals() else print("\n".join(rows))

_ = (LOGS / "part5_run.json").write_text(json.dumps({"targets": TARGETS5, "runs": runs5,
                                                     "versions": {"superseded": vrep["superseded"], "future": vrep["future"],
                                                                  "rows": vrep["rows"]}},
                                                    indent=1, ensure_ascii=False, default=str), encoding="utf-8")
for k in CONFIGS5:
    print(k, "spend:", runs5[k]["spend"])
for q in DQ:
    print(f"\n{q['id']} ({q['type']}) · {q['question']}\n{RS[LAST][q['id']]['answer']}")

The pick of cells 2 and 3 is the Part 4 winner itself, so it runs once.
| Metric | Target (accounts payable) | Part 4 winner |
|---|---|---:|
| Configuration |  | structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10 |
| Retrieval recall@5 | ≥ 90% | 100% (n=10) |
| Recall@1 / @3, MRR | report | 70% / 90%, 0.82 |
| Answer correct (LLM judge) | ≥ 90% | 92% (11/12) |
| Payment-status and permission answers | all right | 2/2 |
| Exact-fact check | report | 92% (n=12) |
| Judge vs exact check | report | agree 100%, kappa 1.0 |
| Citation correct | ≥ 95% | 100% |
| Confidential leaks | 0 | 0 |
| Search p50 / p95 | p95 ≤ 200 ms | 135 / 145 ms |
| Time to answer p50 / p95 | p95 ≤ 10 s | 5.2 / 7.5 s |
| Cost per question | report | $0.0047 |
| Id | Type | Question | Gold rank (Part 4 winner) | Answer (Part 4 winner) | Withheld notice | Judge's reason (Part 4 winner) |
|---|---|---|---:|---|---|---|
| D-01 | single_lookup | need our insurance policy no. for the claim on stuff damaged on the 

In [9]:
# A3 · Part 5 · 5. Rule 12: the check that runs after every change, without a key (part5/check_retrieval.py). It
# rebuilds the index with the configuration cells 2 and 3 picked and exits with 1 if recall@5, the permission filter or the
# version marks regress. Here it runs in a fresh Python process, as a CI job would.
import subprocess

res = subprocess.run([sys.executable, str(PART5 / "check_retrieval.py"), "--latency"], cwd=A3, capture_output=True,
                     text=True, encoding="utf-8")
print(res.stdout[-2000:] or res.stderr[-2000:])
print("exit code:", res.returncode)

configuration: structured-structure-400-o15|bge-small|hybrid|rr1|qr0|k10
PASS  recall@5 >= 90% on 10 questions (got 100%)
PASS  no confidential chunk returned to a non-HR role (leaked on: none)
PASS  each document with several versions has exactly one version in force
PASS  search p95 <= 200 ms (got 130 ms)

exit code: 0
